# AI Training Project

**Muc tieu**: Hoc cach train mot AI model tu goc den ngon.

**Bai toan dau tien**: Du doan `y = 2x + 1`

---

## Roadmap

```
Phase 1  ->  Environment         <- BAN DANG O DAY
Phase 2  ->  Tensor
Phase 3  ->  Dataset
Phase 4  ->  Model
Phase 5  ->  Training
Phase 6  ->  Evaluation
Phase 7  ->  Checkpoint
Phase 8  ->  Stability
Phase 9  ->  Experiment
Phase 10 ->  Scaling
Phase 11 ->  Advanced AI
```

---

## Phase 1 - Environment

Muc tieu cua phase nay:
- Hieu Google Colab va Google Drive hoat dong nhu the nao
- Kiem tra Python, PyTorch, CUDA, GPU
- Mount Google Drive de luu du lieu an toan
- Tao cau truc thu muc project
- Hieu Random Seed va tai sao no quan trong

In [ ]:
# ============================================================
# SETUP: Auto-create src/ and configs/ modules if not present
# Run this cell if you uploaded only AI_Training.ipynb to Colab
# ============================================================
import os
import sys

os.makedirs('src', exist_ok=True)
os.makedirs('configs', exist_ok=True)
os.makedirs('data/raw/dataset_v1', exist_ok=True)
os.makedirs('data/processed/train', exist_ok=True)
os.makedirs('data/processed/validation', exist_ok=True)
os.makedirs('data/processed/test', exist_ok=True)
os.makedirs('models/checkpoints', exist_ok=True)
os.makedirs('models/final', exist_ok=True)
os.makedirs('plots', exist_ok=True)
os.makedirs('experiments', exist_ok=True)

if '.' not in sys.path:
    sys.path.insert(0, '.')

# Embedded source files dictionary
code_files = {
    "src/advanced.py": "\"\"\"\nadvanced.py - Advanced AI foundations and scaling modules (Phases 10 & 11).\nDemonstrates:\n1. Non-linear Polynomial / Multi-feature Math Modeling with Neural Networks (MLP).\n2. Large Dataset scaling (10,000+ samples).\n3. Architecture definitions for Deep Learning extensions (CNN, Transformer, Math Solver).\n\"\"\"\n\nimport os\nimport torch\nimport torch.nn as nn\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom src.utils import set_seed, get_device\nfrom src.model import MLPRegressionModel\nfrom src.evaluate import evaluate_model, plot_predictions\n\n\nclass QuadraticEquationDataset(torch.utils.data.Dataset):\n    \"\"\"\n    Non-linear dataset: y = 2x^2 - 3x + 1 + noise\n    Demonstrates the transition from Linear Regression to Deep Neural Networks.\n    \"\"\"\n    def __init__(self, num_samples: int = 2000, seed: int = 42):\n        torch.manual_seed(seed)\n        self.x = torch.empty(num_samples, 1).uniform_(-5.0, 5.0)\n        noise = torch.randn(num_samples, 1) * 0.8\n        self.y = 2.0 * (self.x ** 2) - 3.0 * self.x + 1.0 + noise\n\n    def __len__(self):\n        return len(self.x)\n\n    def __getitem__(self, idx):\n        return self.x[idx], self.y[idx]\n\n\ndef train_advanced_nn_experiment(project_root: str = \".\", epochs: int = 80, lr: float = 0.01):\n    \"\"\"\n    Train Multi-Layer Perceptron (MLP) on a non-linear quadratic function.\n    \"\"\"\n    set_seed(42)\n    device = get_device()\n    exp_dir = os.path.join(project_root, \"experiments\", \"experiment_002_mlp\")\n    os.makedirs(exp_dir, exist_ok=True)\n\n    print(\"=\" * 65)\n    print(\"      PHASE 11: ADVANCED AI EXPERIMENT (NON-LINEAR MLP)\")\n    print(\"=\" * 65)\n\n    full_dataset = QuadraticEquationDataset(num_samples=3000, seed=42)\n    train_size = int(0.7 * len(full_dataset))\n    val_size = int(0.15 * len(full_dataset))\n    test_size = len(full_dataset) - train_size - val_size\n\n    train_data, val_data, test_data = torch.utils.data.random_split(\n        full_dataset, [train_size, val_size, test_size]\n    )\n\n    train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True)\n    val_loader = torch.utils.data.DataLoader(val_data, batch_size=32, shuffle=False)\n    test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle=False)\n\n    model = MLPRegressionModel(input_size=1, hidden_dim1=64, hidden_dim2=32, output_size=1).to(device)\n    criterion = nn.MSELoss()\n    optimizer = torch.optim.Adam(model.parameters(), lr=lr)\n\n    print(f\"MLP Model Architecture:\\n{model}\")\n    print(f\"Total Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}\")\n\n    train_losses, val_losses = [], []\n    for epoch in range(1, epochs + 1):\n        model.train()\n        total_loss = 0.0\n        for x_b, y_b in train_loader:\n            x_b, y_b = x_b.to(device), y_b.to(device)\n            pred = model(x_b)\n            loss = criterion(pred, y_b)\n\n            optimizer.zero_grad()\n            loss.backward()\n            optimizer.step()\n\n            total_loss += loss.item() * len(x_b)\n\n        epoch_loss = total_loss / len(train_data)\n        val_loss, val_mae, val_r2, _, _ = evaluate_model(model, val_loader, criterion, device)\n\n        train_losses.append(epoch_loss)\n        val_losses.append(val_loss)\n\n        if epoch % 10 == 0 or epoch == 1:\n            print(f\"Epoch [{epoch:2d}/{epochs:2d}] | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val R\u00b2: {val_r2:.4f}\")\n\n    # Plot results\n    plt.figure(figsize=(10, 5))\n    plt.plot(range(1, epochs + 1), train_losses, label=\"Train Loss\", color=\"royalblue\")\n    plt.plot(range(1, epochs + 1), val_losses, label=\"Val Loss\", color=\"darkorange\")\n    plt.title(\"MLP Training & Validation Loss on Non-linear Function (y = 2x\u00b2 - 3x + 1)\")\n    plt.xlabel(\"Epoch\")\n    plt.ylabel(\"MSE Loss\")\n    plt.legend()\n    plt.grid(True, alpha=0.3)\n    plt_path = os.path.join(exp_dir, \"loss_curve_mlp.png\")\n    plt.savefig(plt_path, dpi=150, bbox_inches=\"tight\")\n    plt.close()\n\n    # Save model\n    model_path = os.path.join(exp_dir, \"mlp_model.pth\")\n    torch.save(model.state_dict(), model_path)\n    print(f\"Advanced MLP Model trained and saved to {model_path}\")\n    print(f\"Loss plot saved to {plt_path}\")\n\n    # Evaluate test\n    test_loss, test_mae, test_r2, _, _ = evaluate_model(model, test_loader, criterion, device)\n    print(f\"MLP Test R\u00b2 Score: {test_r2:.4f} | Test MAE: {test_mae:.4f}\")\n    return test_r2\n",
    "src/checkpoint.py": "\"\"\"\ncheckpoint.py - Checkpoint management, saving/loading weights, and resume training utilities.\n\"\"\"\n\nimport os\nimport torch\nimport torch.nn as nn\nimport torch.optim as optim\nfrom typing import Optional, Tuple, Dict, Any\n\n\ndef save_checkpoint(\n    path: str,\n    epoch: int,\n    model: nn.Module,\n    optimizer: optim.Optimizer,\n    scheduler: Optional[Any],\n    loss: float,\n    best_val_loss: float,\n    config: dict\n) -> None:\n    \"\"\"\n    Save complete training state checkpoint to disk.\n    \n    Args:\n        path: Filepath where checkpoint .pth will be saved.\n        epoch: Current training epoch.\n        model: PyTorch model instance.\n        optimizer: PyTorch optimizer instance.\n        scheduler: PyTorch learning rate scheduler (optional).\n        loss: Loss value at current epoch.\n        best_val_loss: Lowest validation loss observed so far.\n        config: Experiment configuration dictionary.\n    \"\"\"\n    os.makedirs(os.path.dirname(path), exist_ok=True)\n    checkpoint = {\n        \"epoch\": epoch,\n        \"model_state_dict\": model.state_dict(),\n        \"optimizer_state_dict\": optimizer.state_dict(),\n        \"scheduler_state_dict\": scheduler.state_dict() if scheduler is not None else None,\n        \"loss\": loss,\n        \"best_val_loss\": best_val_loss,\n        \"config\": config,\n    }\n    torch.save(checkpoint, path)\n    print(f\"-> Checkpoint saved at epoch {epoch}: {path}\")\n\n\ndef load_checkpoint(\n    path: str,\n    model: nn.Module,\n    optimizer: Optional[optim.Optimizer] = None,\n    scheduler: Optional[Any] = None,\n    device: Optional[torch.device] = None\n) -> Tuple[int, float, float, Dict[str, Any]]:\n    \"\"\"\n    Load model weights, optimizer state, and training metadata from a checkpoint.\n    \n    Returns:\n        tuple: (start_epoch, last_loss, best_val_loss, config)\n    \"\"\"\n    if not os.path.exists(path):\n        raise FileNotFoundError(f\"Checkpoint file not found: {path}\")\n\n    map_location = device if device is not None else \"cpu\"\n    checkpoint = torch.load(path, map_location=map_location, weights_only=False)\n\n    model.load_state_dict(checkpoint[\"model_state_dict\"])\n\n    if optimizer is not None and \"optimizer_state_dict\" in checkpoint and checkpoint[\"optimizer_state_dict\"] is not None:\n        optimizer.load_state_dict(checkpoint[\"optimizer_state_dict\"])\n\n    if scheduler is not None and \"scheduler_state_dict\" in checkpoint and checkpoint[\"scheduler_state_dict\"] is not None:\n        scheduler.load_state_dict(checkpoint[\"scheduler_state_dict\"])\n\n    start_epoch = checkpoint.get(\"epoch\", 0) + 1\n    last_loss = checkpoint.get(\"loss\", float(\"inf\"))\n    best_val_loss = checkpoint.get(\"best_val_loss\", float(\"inf\"))\n    config = checkpoint.get(\"config\", {})\n\n    print(f\"-> Successfully loaded checkpoint from {path} (resuming from epoch {start_epoch})\")\n    return start_epoch, last_loss, best_val_loss, config\n\n\ndef save_best_model(path: str, model: nn.Module, config: dict, val_loss: float) -> None:\n    \"\"\"\n    Save best model weights and metadata when validation loss reaches a new minimum.\n    \"\"\"\n    os.makedirs(os.path.dirname(path), exist_ok=True)\n    payload = {\n        \"model_state_dict\": model.state_dict(),\n        \"config\": config,\n        \"val_loss\": val_loss,\n    }\n    torch.save(payload, path)\n    print(f\"\u2605 Saved Best Model (val_loss: {val_loss:.6f}) -> {path}\")\n\n\ndef save_final_model(path: str, model: nn.Module, config: dict) -> None:\n    \"\"\"\n    Save final model after complete training run.\n    \"\"\"\n    os.makedirs(os.path.dirname(path), exist_ok=True)\n    payload = {\n        \"model_state_dict\": model.state_dict(),\n        \"config\": config,\n    }\n    torch.save(payload, path)\n    print(f\"\u2714 Final Model saved -> {path}\")\n",
    "src/dataset.py": "\"\"\"\ndataset.py - Dataset generation, custom PyTorch Dataset class, and DataLoader utilities.\n\"\"\"\n\nimport os\nimport json\nimport yaml\nfrom datetime import datetime\nimport torch\nfrom torch.utils.data import Dataset, DataLoader\nimport matplotlib.pyplot as plt\n\n\nclass EquationDataset(Dataset):\n    \"\"\"\n    PyTorch Dataset wrapper for equation tensors (x, y).\n    \"\"\"\n    def __init__(self, x: torch.Tensor, y: torch.Tensor):\n        self.x = x.float()\n        self.y = y.float()\n\n    def __len__(self) -> int:\n        return len(self.x)\n\n    def __getitem__(self, idx: int):\n        return self.x[idx], self.y[idx]\n\n\ndef create_dataloaders(\n    project_root: str = \".\",\n    batch_size: int = 32,\n    shuffle_train: bool = True\n):\n    \"\"\"\n    Load saved train/val/test splits and construct PyTorch DataLoaders.\n    \"\"\"\n    train_path = os.path.join(project_root, \"data\", \"processed\", \"train\", \"train.pt\")\n    val_path = os.path.join(project_root, \"data\", \"processed\", \"validation\", \"val.pt\")\n    test_path = os.path.join(project_root, \"data\", \"processed\", \"test\", \"test.pt\")\n\n    if not (os.path.exists(train_path) and os.path.exists(val_path) and os.path.exists(test_path)):\n        generate_and_save_dataset(project_root=project_root)\n\n    train_data = torch.load(train_path, weights_only=True)\n    val_data = torch.load(val_path, weights_only=True)\n    test_data = torch.load(test_path, weights_only=True)\n\n    train_dataset = EquationDataset(train_data[\"x\"], train_data[\"y\"])\n    val_dataset = EquationDataset(val_data[\"x\"], val_data[\"y\"])\n    test_dataset = EquationDataset(test_data[\"x\"], test_data[\"y\"])\n\n    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=shuffle_train)\n    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)\n    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)\n\n    return train_loader, val_loader, test_loader, (train_dataset, val_dataset, test_dataset)\n\n\ndef generate_and_save_dataset(project_root: str = \".\", version: str = \"v1\", num_samples: int = 1000, seed: int = 42):\n    \"\"\"\n    Generate synthetic dataset for y = 2x + 1 + noise and split into train/val/test sets.\n    \"\"\"\n    # Fix seed for dataset generation\n    torch.manual_seed(seed)\n\n    config_path = os.path.join(project_root, \"configs\", \"config_v1.yaml\")\n    if os.path.exists(config_path):\n        with open(config_path, \"r\", encoding=\"utf-8\") as f:\n            cfg = yaml.safe_load(f)\n            train_ratio = cfg.get(\"data\", {}).get(\"train_ratio\", 0.70)\n            val_ratio = cfg.get(\"data\", {}).get(\"validation_ratio\", 0.15)\n            test_ratio = cfg.get(\"data\", {}).get(\"test_ratio\", 0.15)\n    else:\n        train_ratio, val_ratio, test_ratio = 0.70, 0.15, 0.15\n\n    raw_dir = os.path.join(project_root, \"data\", \"raw\", f\"dataset_{version}\")\n    train_dir = os.path.join(project_root, \"data\", \"processed\", \"train\")\n    val_dir = os.path.join(project_root, \"data\", \"processed\", \"validation\")\n    test_dir = os.path.join(project_root, \"data\", \"processed\", \"test\")\n    plots_dir = os.path.join(project_root, \"plots\")\n\n    for d in [raw_dir, train_dir, val_dir, test_dir, plots_dir]:\n        os.makedirs(d, exist_ok=True)\n\n    print(\"=\" * 55)\n    print(f\"Generating Dataset {version} ({num_samples} samples)...\")\n    print(\"=\" * 55)\n\n    x = torch.empty(num_samples, 1).uniform_(-10.0, 10.0)\n    noise = torch.randn(num_samples, 1) * 0.5\n    y = 2.0 * x + 1.0 + noise\n\n    # Save raw dataset and metadata\n    dataset_pt_path = os.path.join(raw_dir, \"dataset.pt\")\n    torch.save({\"x\": x, \"y\": y}, dataset_pt_path)\n\n    metadata = {\n        \"version\": version,\n        \"formula\": \"y = 2x + 1 + noise\",\n        \"num_samples\": num_samples,\n        \"x_range\": [-10, 10],\n        \"noise_std\": 0.5,\n        \"seed\": seed,\n        \"x_dtype\": str(x.dtype),\n        \"y_dtype\": str(y.dtype),\n        \"created_at\": datetime.now().isoformat(),\n    }\n    with open(os.path.join(raw_dir, \"metadata.json\"), \"w\", encoding=\"utf-8\") as f:\n        json.dump(metadata, f, indent=2)\n\n    # Plot raw data\n    plt.figure(figsize=(8, 5))\n    plt.scatter(x.numpy(), y.numpy(), alpha=0.3, s=15, color=\"blue\", label=\"Samples (with noise)\")\n    plt.plot([-10, 10], [2 * (-10) + 1, 2 * 10 + 1], \"r--\", linewidth=2, label=\"True line: y = 2x + 1\")\n    plt.title(f\"Dataset {version}: y = 2x + 1 + noise (N={num_samples})\")\n    plt.xlabel(\"x\")\n    plt.ylabel(\"y\")\n    plt.legend()\n    plt.grid(True, alpha=0.3)\n    raw_plot_path = os.path.join(plots_dir, f\"dataset_{version}.png\")\n    plt.savefig(raw_plot_path, dpi=150, bbox_inches=\"tight\")\n    plt.close()\n\n    # Split dataset\n    n_train = int(num_samples * train_ratio)\n    n_val = int(num_samples * val_ratio)\n    n_test = num_samples - n_train - n_val\n\n    indices = torch.randperm(num_samples)\n    train_idx = indices[:n_train]\n    val_idx = indices[n_train:n_train + n_val]\n    test_idx = indices[n_train + n_val:]\n\n    x_train, y_train = x[train_idx], y[train_idx]\n    x_val, y_val = x[val_idx], y[val_idx]\n    x_test, y_test = x[test_idx], y[test_idx]\n\n    torch.save({\"x\": x_train, \"y\": y_train}, os.path.join(train_dir, \"train.pt\"))\n    torch.save({\"x\": x_val, \"y\": y_val}, os.path.join(val_dir, \"val.pt\"))\n    torch.save({\"x\": x_test, \"y\": y_test}, os.path.join(test_dir, \"test.pt\"))\n\n    split_metadata = {\n        \"dataset_version\": version,\n        \"total_samples\": num_samples,\n        \"splits\": {\n            \"train\": {\"samples\": n_train, \"ratio\": train_ratio},\n            \"validation\": {\"samples\": n_val, \"ratio\": val_ratio},\n            \"test\": {\"samples\": n_test, \"ratio\": round(n_test / num_samples, 4)},\n        },\n        \"seed\": seed,\n    }\n    with open(os.path.join(project_root, \"data\", \"processed\", \"split_metadata.json\"), \"w\", encoding=\"utf-8\") as f:\n        json.dump(split_metadata, f, indent=2)\n\n    # Plot splits\n    plt.figure(figsize=(9, 5))\n    plt.scatter(x_train.numpy(), y_train.numpy(), alpha=0.4, s=15, color=\"blue\", label=f\"Train ({n_train})\")\n    plt.scatter(x_val.numpy(), y_val.numpy(), alpha=0.6, s=20, color=\"orange\", label=f\"Validation ({n_val})\")\n    plt.scatter(x_test.numpy(), y_test.numpy(), alpha=0.8, s=25, color=\"green\", label=f\"Test ({n_test})\")\n    plt.title(f\"Dataset {version} Splits (Train={n_train}, Val={n_val}, Test={n_test})\")\n    plt.xlabel(\"x\")\n    plt.ylabel(\"y\")\n    plt.legend()\n    plt.grid(True, alpha=0.3)\n    split_plot_path = os.path.join(plots_dir, f\"dataset_{version}_split.png\")\n    plt.savefig(split_plot_path, dpi=150, bbox_inches=\"tight\")\n    plt.close()\n\n    print(f\"Splits saved: Train ({n_train}) | Val ({n_val}) | Test ({n_test})\")\n    print(f\"Saved plots to {plots_dir}\")\n\n\nif __name__ == \"__main__\":\n    generate_and_save_dataset(project_root=\".\")\n",
    "src/evaluate.py": "\"\"\"\nevaluate.py - Evaluation functions, metrics calculation (MSE, MAE, R\u00b2), and visualization routines.\n\"\"\"\n\nimport os\nimport torch\nimport torch.nn as nn\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score\n\n\ndef evaluate_model(model: nn.Module, data_loader, criterion: nn.Module, device: torch.device):\n    \"\"\"\n    Evaluate model performance on a dataset using model.eval() and torch.no_grad().\n    \n    Returns:\n        tuple: (avg_loss, mae, r2, y_true_all, y_pred_all)\n    \"\"\"\n    model.eval()  # Switch to evaluation mode\n    total_loss = 0.0\n    all_y_true = []\n    all_y_pred = []\n\n    with torch.no_grad():  # Disable gradient computation to save memory & compute\n        for x_batch, y_batch in data_loader:\n            x_batch = x_batch.to(device)\n            y_batch = y_batch.to(device)\n\n            y_pred = model(x_batch)\n            loss = criterion(y_pred, y_batch)\n\n            total_loss += loss.item() * len(x_batch)\n            all_y_true.append(y_batch.cpu().numpy())\n            all_y_pred.append(y_pred.cpu().numpy())\n\n    total_samples = len(data_loader.dataset)\n    avg_loss = total_loss / total_samples if total_samples > 0 else 0.0\n\n    y_true_all = np.vstack(all_y_true).squeeze()\n    y_pred_all = np.vstack(all_y_pred).squeeze()\n\n    mae = float(mean_absolute_error(y_true_all, y_pred_all))\n    r2 = float(r2_score(y_true_all, y_pred_all))\n\n    return avg_loss, mae, r2, y_true_all, y_pred_all\n\n\ndef test_unseen_points(model: nn.Module, device: torch.device, custom_points=None):\n    \"\"\"\n    Test model on unseen test coordinates and print diagnostic comparison table.\n    \"\"\"\n    if custom_points is None:\n        custom_points = [11.0, 15.0, 20.0, 50.0, -15.0, 0.0]\n\n    model.eval()\n    x_tensor = torch.tensor([[pt] for pt in custom_points], dtype=torch.float32).to(device)\n\n    with torch.no_grad():\n        preds = model(x_tensor).cpu().numpy().flatten()\n\n    results = []\n    print(\"\\n\" + \"=\" * 70)\n    print(\"                 UNSEEN TEST SAMPLES EVALUATION\")\n    print(\"=\" * 70)\n    print(f\"{'Input (x)':<12} | {'Expected (2x+1)':<18} | {'Prediction':<16} | {'Abs Error':<12}\")\n    print(\"-\" * 70)\n    for x_val, pred in zip(custom_points, preds):\n        expected = 2.0 * x_val + 1.0\n        err = abs(pred - expected)\n        results.append({\"x\": x_val, \"expected\": expected, \"prediction\": pred, \"abs_error\": err})\n        print(f\"{x_val:<12.2f} | {expected:<18.4f} | {pred:<16.4f} | {err:<12.4f}\")\n    print(\"=\" * 70)\n    return results\n\n\ndef plot_training_history(metrics_csv_path: str, save_path: str = None):\n    \"\"\"\n    Plot Train/Val Loss, MAE, R\u00b2, and Learning Rate curves from metrics.csv.\n    \"\"\"\n    if not os.path.exists(metrics_csv_path):\n        print(f\"Metrics file not found: {metrics_csv_path}\")\n        return\n\n    df = pd.read_csv(metrics_csv_path)\n    if df.empty:\n        return\n\n    epochs = df[\"epoch\"]\n    \n    fig, axes = plt.subplots(2, 2, figsize=(14, 10))\n\n    # 1. Loss Curve\n    axes[0, 0].plot(epochs, df[\"train_loss\"], label=\"Train Loss (MSE)\", color=\"royalblue\", lw=2)\n    if \"val_loss\" in df.columns:\n        axes[0, 0].plot(epochs, df[\"val_loss\"], label=\"Val Loss (MSE)\", color=\"darkorange\", lw=2, linestyle=\"--\")\n    axes[0, 0].set_title(\"Training & Validation Loss\")\n    axes[0, 0].set_xlabel(\"Epoch\")\n    axes[0, 0].set_ylabel(\"MSE Loss\")\n    axes[0, 0].legend()\n    axes[0, 0].grid(True, alpha=0.3)\n\n    # 2. MAE Curve\n    if \"val_mae\" in df.columns:\n        axes[0, 1].plot(epochs, df[\"val_mae\"], label=\"Validation MAE\", color=\"crimson\", lw=2)\n        axes[0, 1].set_title(\"Validation MAE (Mean Absolute Error)\")\n        axes[0, 1].set_xlabel(\"Epoch\")\n        axes[0, 1].set_ylabel(\"MAE\")\n        axes[0, 1].legend()\n        axes[0, 1].grid(True, alpha=0.3)\n\n    # 3. R2 Score Curve\n    if \"val_r2\" in df.columns:\n        axes[1, 0].plot(epochs, df[\"val_r2\"], label=\"Validation R\u00b2 Score\", color=\"forestgreen\", lw=2)\n        axes[1, 0].set_title(\"Validation R\u00b2 Score (Target = 1.0)\")\n        axes[1, 0].set_xlabel(\"Epoch\")\n        axes[1, 0].set_ylabel(\"R\u00b2 Score\")\n        axes[1, 0].legend()\n        axes[1, 0].grid(True, alpha=0.3)\n\n    # 4. Learning Rate / Gradient Norm Curve\n    if \"lr\" in df.columns:\n        axes[1, 1].plot(epochs, df[\"lr\"], label=\"Learning Rate\", color=\"purple\", lw=2)\n        axes[1, 1].set_title(\"Learning Rate Schedule\")\n        axes[1, 1].set_xlabel(\"Epoch\")\n        axes[1, 1].set_ylabel(\"LR\")\n        axes[1, 1].set_yscale(\"log\")\n        axes[1, 1].legend()\n        axes[1, 1].grid(True, alpha=0.3)\n\n    plt.tight_layout()\n    if save_path:\n        os.makedirs(os.path.dirname(save_path), exist_ok=True)\n        plt.savefig(save_path, dpi=150, bbox_inches=\"tight\")\n        print(f\"Saved training curves to {save_path}\")\n    plt.close()\n\n\ndef plot_predictions(model: nn.Module, device: torch.device, data_loader, save_path: str = None):\n    \"\"\"\n    Plot model predictions vs actual ground truth line.\n    \"\"\"\n    model.eval()\n    x_list, y_list, pred_list = [], [], []\n\n    with torch.no_grad():\n        for x_b, y_b in data_loader:\n            x_b_dev = x_b.to(device)\n            p_b = model(x_b_dev)\n            x_list.append(x_b.cpu().numpy())\n            y_list.append(y_b.cpu().numpy())\n            pred_list.append(p_b.cpu().numpy())\n\n    x_all = np.vstack(x_list).flatten()\n    y_all = np.vstack(y_list).flatten()\n    preds_all = np.vstack(pred_list).flatten()\n\n    plt.figure(figsize=(9, 6))\n    plt.scatter(x_all, y_all, alpha=0.4, color=\"royalblue\", label=\"Actual Data (with noise)\", s=20)\n    \n    sort_idx = np.argsort(x_all)\n    plt.plot(x_all[sort_idx], preds_all[sort_idx], color=\"crimson\", linewidth=2.5, label=\"Model Prediction\")\n    plt.plot(x_all[sort_idx], 2.0 * x_all[sort_idx] + 1.0, \"k--\", linewidth=1.5, label=\"Ideal True Line (y = 2x + 1)\")\n\n    plt.title(\"Model Prediction vs Actual Data\", fontsize=14)\n    plt.xlabel(\"x\", fontsize=12)\n    plt.ylabel(\"y\", fontsize=12)\n    plt.legend(fontsize=11)\n    plt.grid(True, alpha=0.3)\n\n    if save_path:\n        os.makedirs(os.path.dirname(save_path), exist_ok=True)\n        plt.savefig(save_path, dpi=150, bbox_inches=\"tight\")\n        print(f\"Saved prediction plot to {save_path}\")\n    plt.close()\n",
    "src/model.py": "\"\"\"\nmodel.py - Model definitions: Linear Regression and Multi-Layer Perceptron (MLP).\n\"\"\"\n\nimport torch\nimport torch.nn as nn\n\n\nclass LinearRegressionModel(nn.Module):\n    \"\"\"\n    Linear Regression model: y = x * weight + bias\n    \n    Args:\n        input_size (int): Number of input features (default: 1)\n        output_size (int): Number of output targets (default: 1)\n    \"\"\"\n    def __init__(self, input_size: int = 1, output_size: int = 1):\n        super(LinearRegressionModel, self).__init__()\n        self.linear = nn.Linear(in_features=input_size, out_features=output_size)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.linear(x)\n\n\nclass MLPRegressionModel(nn.Module):\n    \"\"\"\n    Multi-Layer Perceptron (Neural Network) with non-linear activations (ReLU).\n    Used for Phase 10 & 11 to approximate non-linear formulas and complex math functions.\n    \n    Architecture:\n        Input Layer -> Hidden Layer 1 (64) -> ReLU -> Hidden Layer 2 (32) -> ReLU -> Output Layer\n    \"\"\"\n    def __init__(self, input_size: int = 1, hidden_dim1: int = 64, hidden_dim2: int = 32, output_size: int = 1):\n        super(MLPRegressionModel, self).__init__()\n        self.net = nn.Sequential(\n            nn.Linear(input_size, hidden_dim1),\n            nn.ReLU(),\n            nn.Linear(hidden_dim1, hidden_dim2),\n            nn.ReLU(),\n            nn.Linear(hidden_dim2, output_size)\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n\ndef get_model_parameters(model: nn.Module) -> dict:\n    \"\"\"\n    Extract weights and bias values from a linear model or summarize parameter count.\n    \"\"\"\n    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    res = {\"total_parameters\": total_params}\n    if hasattr(model, \"linear\"):\n        weight = model.linear.weight.data.clone()\n        bias = model.linear.bias.data.clone()\n        res.update({\n            \"weight\": weight,\n            \"bias\": bias,\n            \"weight_val\": weight.item() if weight.numel() == 1 else weight.tolist(),\n            \"bias_val\": bias.item() if bias.numel() == 1 else bias.tolist(),\n        })\n    return res\n\n\nif __name__ == \"__main__\":\n    torch.manual_seed(42)\n    lin_model = LinearRegressionModel()\n    mlp_model = MLPRegressionModel()\n    \n    x_test = torch.tensor([[1.0], [2.0], [10.0]])\n    print(\"Linear model output:\", lin_model(x_test).squeeze())\n    print(\"MLP model output:\", mlp_model(x_test).squeeze())\n",
    "src/train.py": "\"\"\"\ntrain.py - Core training pipeline for AI Training Project.\nImplements complete training loop, validation, optimizer, scheduler, gradient clipping,\nearly stopping, metrics logging (metrics.csv & train.log), and checkpointing.\n\"\"\"\n\nimport os\nimport sys\nimport csv\nimport shutil\nimport torch\nimport torch.nn as nn\nimport torch.optim as optim\n\nfrom src.utils import set_seed, get_device, load_config, save_config\nfrom src.dataset import create_dataloaders\nfrom src.model import LinearRegressionModel, MLPRegressionModel, get_model_parameters\nfrom src.evaluate import evaluate_model, test_unseen_points, plot_training_history, plot_predictions\nfrom src.checkpoint import save_checkpoint, load_checkpoint, save_best_model, save_final_model\n\n\ndef train_model(\n    config_path: str = \"configs/config_v1.yaml\",\n    experiment_name: str = \"experiment_001\",\n    project_root: str = \".\",\n    resume_checkpoint_path: str = None\n):\n    \"\"\"\n    Execute full training pipeline based on configuration.\n    \"\"\"\n    # 1. Load config\n    config = load_config(os.path.join(project_root, config_path))\n    seed = config.get(\"seed\", 42)\n    set_seed(seed)\n    device = get_device()\n\n    # 2. Setup Experiment directory and immutable config copy\n    exp_dir = os.path.join(project_root, \"experiments\", experiment_name)\n    os.makedirs(exp_dir, exist_ok=True)\n    immutable_config_path = os.path.join(exp_dir, \"config.yaml\")\n    save_config(config, immutable_config_path)\n\n    log_path = os.path.join(exp_dir, \"train.log\")\n    metrics_csv_path = os.path.join(exp_dir, \"metrics.csv\")\n    checkpoints_dir = os.path.join(project_root, \"models\", \"checkpoints\")\n    final_model_dir = os.path.join(project_root, \"models\", \"final\")\n    os.makedirs(checkpoints_dir, exist_ok=True)\n    os.makedirs(final_model_dir, exist_ok=True)\n\n    # 3. Create DataLoaders\n    batch_size = config.get(\"training\", {}).get(\"batch_size\", 32)\n    train_loader, val_loader, test_loader, _ = create_dataloaders(\n        project_root=project_root,\n        batch_size=batch_size,\n        shuffle_train=True\n    )\n\n    # 4. Instantiate Model\n    model_type = config.get(\"model\", {}).get(\"type\", \"linear\")\n    input_size = config.get(\"model\", {}).get(\"input_size\", 1)\n    output_size = config.get(\"model\", {}).get(\"output_size\", 1)\n\n    if model_type == \"mlp\":\n        model = MLPRegressionModel(input_size=input_size, output_size=output_size).to(device)\n    else:\n        model = LinearRegressionModel(input_size=input_size, output_size=output_size).to(device)\n\n    # 5. Loss Function and Optimizer\n    criterion = nn.MSELoss()\n    lr = config.get(\"training\", {}).get(\"learning_rate\", 0.001)\n    optimizer_name = config.get(\"training\", {}).get(\"optimizer\", \"adam\").lower()\n\n    if optimizer_name == \"sgd\":\n        optimizer = optim.SGD(model.parameters(), lr=lr)\n    else:\n        optimizer = optim.Adam(model.parameters(), lr=lr)\n\n    # 6. Learning Rate Scheduler (optional)\n    scheduler_cfg = config.get(\"scheduler\", {})\n    use_scheduler = scheduler_cfg.get(\"use_scheduler\", False)\n    scheduler = None\n    if use_scheduler:\n        scheduler_patience = scheduler_cfg.get(\"scheduler_patience\", 5)\n        scheduler_factor = scheduler_cfg.get(\"scheduler_factor\", 0.5)\n        scheduler = optim.lr_scheduler.ReduceLROnPlateau(\n            optimizer, mode=\"min\", patience=scheduler_patience, factor=scheduler_factor\n        )\n\n    # 7. Checkpoint / Resume Handling\n    start_epoch = 1\n    best_val_loss = float(\"inf\")\n    epochs_no_improve = 0\n    early_stopping_patience = config.get(\"training\", {}).get(\"early_stopping_patience\", 10)\n    clip_norm = config.get(\"training\", {}).get(\"gradient_clip_norm\", None)\n    total_epochs = config.get(\"training\", {}).get(\"epochs\", 100)\n\n    if resume_checkpoint_path and os.path.exists(resume_checkpoint_path):\n        start_epoch, _, best_val_loss, _ = load_checkpoint(\n            resume_checkpoint_path, model, optimizer, scheduler, device=device\n        )\n\n    # Initialize metrics CSV file if starting from scratch\n    if start_epoch == 1 or not os.path.exists(metrics_csv_path):\n        with open(metrics_csv_path, \"w\", newline=\"\", encoding=\"utf-8\") as f:\n            writer = csv.writer(f)\n            writer.writerow([\"epoch\", \"train_loss\", \"val_loss\", \"val_mae\", \"val_r2\", \"lr\", \"grad_norm\"])\n\n    log_file = open(log_path, \"a\", encoding=\"utf-8\")\n\n    def log_print(msg: str):\n        print(msg)\n        log_file.write(msg + \"\\n\")\n        log_file.flush()\n\n    log_print(\"=\" * 65)\n    log_print(f\"       STARTING TRAINING: {experiment_name.upper()}\")\n    log_print(\"=\" * 65)\n    log_print(f\"Device: {device} | Total Epochs: {total_epochs} | Batch Size: {batch_size} | Initial LR: {lr}\")\n    log_print(f\"Model Architecture:\\n{model}\")\n    log_print(\"-\" * 65)\n\n    # 8. Main Training Loop\n    for epoch in range(start_epoch, total_epochs + 1):\n        model.train()  # Switch to training mode\n        running_train_loss = 0.0\n        last_grad_norm = 0.0\n\n        for x_batch, y_batch in train_loader:\n            x_batch = x_batch.to(device)\n            y_batch = y_batch.to(device)\n\n            # --- FORWARD PASS ---\n            y_pred = model(x_batch)\n\n            # --- LOSS COMPUTATION ---\n            loss = criterion(y_pred, y_batch)\n\n            # --- BACKPROPAGATION ---\n            optimizer.zero_grad()  # Reset gradients from previous step\n            loss.backward()        # Compute dLoss/dWeight & dLoss/dBias\n\n            # --- GRADIENT CLIPPING (Optional) ---\n            if clip_norm is not None:\n                last_grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_norm).item()\n            else:\n                # Compute gradient norm for tracking\n                total_norm = 0.0\n                for p in model.parameters():\n                    if p.grad is not None:\n                        total_norm += p.grad.data.norm(2).item() ** 2\n                last_grad_norm = total_norm ** 0.5\n\n            # --- OPTIMIZER STEP ---\n            optimizer.step()       # Update weights: w = w - lr * grad\n\n            running_train_loss += loss.item() * len(x_batch)\n\n        epoch_train_loss = running_train_loss / len(train_loader.dataset)\n\n        # --- VALIDATION LOOP (torch.no_grad() & model.eval()) ---\n        val_loss, val_mae, val_r2, _, _ = evaluate_model(model, val_loader, criterion, device)\n\n        # --- LR SCHEDULER STEP ---\n        current_lr = optimizer.param_groups[0][\"lr\"]\n        if scheduler is not None:\n            scheduler.step(val_loss)\n\n        # Record metrics to CSV\n        with open(metrics_csv_path, \"a\", newline=\"\", encoding=\"utf-8\") as f:\n            writer = csv.writer(f)\n            writer.writerow([epoch, epoch_train_loss, val_loss, val_mae, val_r2, current_lr, last_grad_norm])\n\n        # Logging periodic progress\n        if epoch % 10 == 0 or epoch == 1 or epoch == total_epochs:\n            params = get_model_parameters(model)\n            if \"weight_val\" in params and \"bias_val\" in params:\n                param_str = f\"w: {params['weight_val']:.4f}, b: {params['bias_val']:.4f}\"\n            else:\n                param_str = f\"params: {params['total_parameters']}\"\n            log_print(\n                f\"Epoch [{epoch:3d}/{total_epochs:3d}] | \"\n                f\"Train Loss: {epoch_train_loss:.6f} | \"\n                f\"Val Loss: {val_loss:.6f} | \"\n                f\"Val MAE: {val_mae:.4f} | \"\n                f\"Val R\u00b2: {val_r2:.4f} | \"\n                f\"{param_str}\"\n            )\n\n        # --- CHECKPOINT SAVING ---\n        if epoch % 20 == 0:\n            ckpt_file = os.path.join(checkpoints_dir, f\"checkpoint_epoch_{epoch}.pth\")\n            save_checkpoint(ckpt_file, epoch, model, optimizer, scheduler, epoch_train_loss, best_val_loss, config)\n\n        # --- BEST MODEL TRACKING & EARLY STOPPING ---\n        if val_loss < best_val_loss:\n            best_val_loss = val_loss\n            epochs_no_improve = 0\n            best_model_path = os.path.join(exp_dir, \"best_model.pth\")\n            ckpt_best_path = os.path.join(checkpoints_dir, \"best_model.pth\")\n            save_best_model(best_model_path, model, config, val_loss)\n            save_best_model(ckpt_best_path, model, config, val_loss)\n        else:\n            epochs_no_improve += 1\n            if epochs_no_improve >= early_stopping_patience:\n                log_print(f\"\u26a0 Early Stopping triggered at epoch {epoch} (No improvement for {early_stopping_patience} epochs).\")\n                break\n\n    log_print(\"-\" * 65)\n    log_print(\"Training complete.\")\n\n    # 9. Save final experiment model\n    final_model_exp_path = os.path.join(exp_dir, \"model.pth\")\n    final_model_global_path = os.path.join(final_model_dir, f\"{experiment_name}_final.pth\")\n    save_final_model(final_model_exp_path, model, config)\n    save_final_model(final_model_global_path, model, config)\n\n    # 10. Generate Training Plots\n    loss_png_exp = os.path.join(exp_dir, \"loss.png\")\n    loss_png_global = os.path.join(project_root, \"plots\", f\"{experiment_name}_loss.png\")\n    plot_training_history(metrics_csv_path, save_path=loss_png_exp)\n    plot_training_history(metrics_csv_path, save_path=loss_png_global)\n\n    pred_png_exp = os.path.join(exp_dir, \"predictions.png\")\n    plot_predictions(model, device, test_loader, save_path=pred_png_exp)\n\n    # 11. Final Test Evaluation\n    log_print(\"\\n\" + \"=\" * 65)\n    log_print(\"               FINAL TEST SET EVALUATION\")\n    log_print(\"=\" * 65)\n    test_loss, test_mae, test_r2, _, _ = evaluate_model(model, test_loader, criterion, device)\n    log_print(f\"Test Loss (MSE): {test_loss:.6f}\")\n    log_print(f\"Test MAE:        {test_mae:.6f}\")\n    log_print(f\"Test R\u00b2 Score:   {test_r2:.6f}\")\n\n    # Unseen points test\n    test_unseen_points(model, device)\n\n    log_file.close()\n    return {\n        \"best_val_loss\": best_val_loss,\n        \"test_loss\": test_loss,\n        \"test_mae\": test_mae,\n        \"test_r2\": test_r2,\n        \"exp_dir\": exp_dir\n    }\n\n\nif __name__ == \"__main__\":\n    train_model()\n",
    "src/utils.py": "\"\"\"\nutils.py - General utilities for AI Training Project.\nIncludes random seed fixing, device detection, config loading/saving, and logging helpers.\n\"\"\"\n\nimport os\nimport random\nimport yaml\nimport numpy as np\nimport torch\n\n\ndef set_seed(seed: int = 42) -> None:\n    \"\"\"\n    Fix random seeds for reproducibility across random, numpy, and PyTorch (CPU & CUDA).\n    \"\"\"\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n        torch.backends.cudnn.deterministic = True\n        torch.backends.cudnn.benchmark = False\n\n\ndef get_device() -> torch.device:\n    \"\"\"\n    Detect available computing hardware (CUDA GPU or CPU) and print diagnostic info.\n    \"\"\"\n    if torch.cuda.is_available():\n        device = torch.device(\"cuda\")\n        print(f\"Device: CUDA ({torch.cuda.get_device_name(0)})\")\n        print(f\"PyTorch Version: {torch.__version__} | CUDA Version: {torch.version.cuda}\")\n    else:\n        device = torch.device(\"cpu\")\n        print(f\"Device: CPU | PyTorch Version: {torch.__version__}\")\n    return device\n\n\ndef load_config(config_path: str) -> dict:\n    \"\"\"\n    Load YAML configuration file into a Python dictionary.\n    \"\"\"\n    if not os.path.exists(config_path):\n        raise FileNotFoundError(f\"Config file not found: {config_path}\")\n    with open(config_path, \"r\", encoding=\"utf-8\") as f:\n        config = yaml.safe_load(f)\n    return config\n\n\ndef save_config(config: dict, dest_path: str) -> None:\n    \"\"\"\n    Save configuration dictionary to a YAML file (used for immutable experiment config).\n    \"\"\"\n    os.makedirs(os.path.dirname(dest_path), exist_ok=True)\n    with open(dest_path, \"w\", encoding=\"utf-8\") as f:\n        yaml.dump(config, f, default_flow_style=False, sort_keys=False)\n",
    "configs/config_v1.yaml": "seed: 42\nmodel:\n  input_size: 1\n  output_size: 1\ntraining:\n  epochs: 100\n  batch_size: 32\n  learning_rate: 0.001\n  early_stopping_patience: 10\n  gradient_clip_norm: null\nscheduler:\n  use_scheduler: false\n  scheduler_type: ReduceLROnPlateau\n  scheduler_patience: 5\n  scheduler_factor: 0.5\ndata:\n  train_ratio: 0.7\n  validation_ratio: 0.15\n  test_ratio: 0.15\n  dataset_version: v1\n  num_samples: 1000\n",
    "configs/config_v2_nn.yaml": "seed: 42\nmodel:\n  type: mlp\n  input_size: 1\n  hidden_dim1: 64\n  hidden_dim2: 32\n  output_size: 1\ntraining:\n  epochs: 100\n  batch_size: 32\n  learning_rate: 0.01\n  optimizer: adam\n  early_stopping_patience: 15\n  gradient_clip_norm: 1.0\nscheduler:\n  use_scheduler: true\n  scheduler_type: ReduceLROnPlateau\n  scheduler_patience: 5\n  scheduler_factor: 0.5\ndata:\n  train_ratio: 0.7\n  validation_ratio: 0.15\n  test_ratio: 0.15\n  dataset_version: v2_nonlinear\n  num_samples: 3000\n"
}

for filepath, content in code_files.items():
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content)

print('✓ Successfully configured src/ modules, configs, and project structure!')


---

## Cell 1 - Kiem tra moi truong

**WHAT**: Cell nay kiem tra toan bo moi truong Google Colab.

**WHY**: Truoc khi bat dau bat ky dieu gi, phai biet:
- Python version la bao nhieu?
- PyTorch version la bao nhieu?
- Co GPU khong?
- CUDA la gi?

**HOW**:
- `sys.version` -> Python version
- `torch.__version__` -> PyTorch version
- `torch.cuda.is_available()` -> True neu co GPU
- `torch.cuda.get_device_name(0)` -> Ten GPU

**WHEN**: Chay cell nay **dau tien** moi khi mo lai notebook tren Colab.

---

### Giai thich cac khai niem:

**GPU (Graphics Processing Unit)**:
GPU ban dau duoc tao ra de render do hoa game. Nhung vi GPU co hang nghin core nho chay song song, no cuc ky phu hop cho viec tinh toan ma tran - thu AI can nhieu nhat.
Thay vi 8 core CPU tinh lan luot, GPU T4 cua Colab co 2,560 core tinh dong thoi.

**CUDA (Compute Unified Device Architecture)**:
CUDA la nen tang cua NVIDIA cho phep lap trinh vien dung GPU de tinh toan (khong chi do hoa). PyTorch dung CUDA de chay cac phep tinh tensor tren GPU.

**PyTorch**:
Thu vien Python de xay dung va train neural network. PyTorch tu dong tinh gradient (dao ham) - thu can thiet de cap nhat weight trong qua trinh hoc.

In [ ]:
# ============================================================
# CELL 1: Check Environment
# Purpose: Verify Python, PyTorch, CUDA, and GPU availability
# Run this cell FIRST every time you open Colab
# ============================================================

import sys
import torch

print("=" * 55)
print("         ENVIRONMENT CHECK")
print("=" * 55)

# --- Python version ---
python_version = sys.version.split(" ")[0]
print(f"Python version : {python_version}")

# --- PyTorch version ---
print(f"PyTorch version: {torch.__version__}")

# --- CUDA availability ---
cuda_available = torch.cuda.is_available()
print(f"CUDA available : {cuda_available}")

# --- CUDA version ---
if cuda_available:
    print(f"CUDA version   : {torch.version.cuda}")
else:
    print("CUDA version   : N/A (no GPU detected)")

# --- Device selection ---
# 'cuda' = GPU, 'cpu' = processor
device = torch.device("cuda" if cuda_available else "cpu")
print(f"Device selected: {device}")

# --- GPU details ---
if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU name       : {gpu_name}")
    print(f"GPU memory     : {total_mem:.2f} GB")
else:
    print("GPU name       : N/A (running on CPU)")
    print("Tip: Go to Runtime -> Change runtime type -> GPU to enable GPU")

print("=" * 55)
print("Environment check complete.")
print("=" * 55)

---

## Cell 2 - Mount Google Drive

**WHAT**: Mount (ket noi) Google Drive vao Colab runtime.

**WHY**: Google Colab la **may tinh di muon**. Khi ban dong tab hoac runtime bi reset (thuong sau 12 gio khong hoat dong), **toan bo du lieu trong `/content/` bien mat**.

Giai phap: Luu moi thu quan trong len **Google Drive** - du lieu tren Drive se ton tai mai mai.

**HOW**: Colab tao mot 'cau noi' (mount point) tai `/content/drive/` - khi ban doc/ghi vao duong dan nay, ban thuc ra dang doc/ghi truc tiep len Google Drive.

```
Google Colab Runtime              Google Drive
---------------------             ----------------
/content/                         MyDrive/
  +-- drive/            <--mount--   +-- AI_Training/
        +-- MyDrive/                       +-- data/
              +-- AI_Training/             +-- models/
                    +-- data/              +-- configs/
                    +-- models/            +-- experiments/
```

**WHEN**: Chay ngay sau Cell 1, truoc khi lam bat ky dieu gi khac.

In [ ]:
# ============================================================
# CELL 2: Mount Google Drive
# Purpose: Connect Google Drive so data is saved permanently
# ============================================================

from google.colab import drive
import os

# Mount Google Drive at /content/drive
# This will open a popup asking for permission - click 'Allow'
drive.mount("/content/drive")

# Verify the mount was successful
if os.path.exists("/content/drive/MyDrive"):
    print("Google Drive mounted successfully.")
    print(f"Mount point: /content/drive/MyDrive")
else:
    print("Mount failed. Please re-run this cell and allow access.")

---

## Cell 3 - Tao cau truc thu muc project

**WHAT**: Tao cac thu muc can thiet cho toan bo project.

**WHY**: Mot project ML tot can to chuc ro rang:
- Khong tron lan data voi code
- Khong tron lan checkpoint voi final model
- Khong tron lan config cua cac experiment khac nhau

**HOW**: Dung `os.makedirs()` voi `exist_ok=True` - neu thu muc da ton tai thi bo qua, khong bao loi.

**WHEN**: Chi can chay mot lan. Cac lan sau, thu muc da co roi.

### Cau truc project:

```
AI_Training/
+-- data/
|   +-- raw/
|   |   +-- dataset_v1/     <- 1,000 samples
|   |   +-- dataset_v2/     <- 10,000 samples (tuong lai)
|   |   +-- dataset_v3/     <- 100,000 samples (tuong lai)
|   +-- processed/
|       +-- train/           <- 70% data
|       +-- validation/      <- 15% data
|       +-- test/            <- 15% data
+-- models/
|   +-- checkpoints/         <- checkpoint_epoch_10.pth, ...
|   +-- final/               <- model_v1.pth, model_v2.pth, ...
+-- configs/                 <- config_v1.yaml, config_v2.yaml
+-- experiments/
|   +-- experiment_001/
|       +-- config.yaml      <- copy tu template, khong sua sau khi train
|       +-- train.log        <- log text
|       +-- metrics.csv      <- epoch, loss, mae, r2
|       +-- loss.png         <- bieu do loss
|       +-- model.pth        <- model cua experiment nay
+-- notebooks/
+-- src/
|   +-- dataset.py
|   +-- model.py
|   +-- train.py
|   +-- evaluate.py
|   +-- checkpoint.py
|   +-- utils.py
+-- plots/
+-- todo.md
+-- README.md
+-- requirements.txt
```

In [ ]:
# ============================================================
# CELL 3: Create Project Directory Structure
# Purpose: Set up all folders needed for the entire project
# ============================================================

import os

# --- Project root on Google Drive ---
import sys
IN_COLAB = 'google.colab' in sys.modules
PROJECT_ROOT = "/content/drive/MyDrive/AI_Training" if IN_COLAB else os.path.abspath(".")

# --- Define all directories to create ---
directories = [
    # Data
    f"{PROJECT_ROOT}/data/raw/dataset_v1",
    f"{PROJECT_ROOT}/data/raw/dataset_v2",
    f"{PROJECT_ROOT}/data/raw/dataset_v3",
    f"{PROJECT_ROOT}/data/processed/train",
    f"{PROJECT_ROOT}/data/processed/validation",
    f"{PROJECT_ROOT}/data/processed/test",

    # Models
    f"{PROJECT_ROOT}/models/checkpoints",
    f"{PROJECT_ROOT}/models/final",

    # Config
    f"{PROJECT_ROOT}/configs",

    # Experiments
    f"{PROJECT_ROOT}/experiments/experiment_001",

    # Source code
    f"{PROJECT_ROOT}/notebooks",
    f"{PROJECT_ROOT}/src",

    # Plots
    f"{PROJECT_ROOT}/plots",
]

# --- Create directories ---
print("Creating project structure...")
print("=" * 55)

for directory in directories:
    os.makedirs(directory, exist_ok=True)
    # Show relative path for cleaner output
    relative_path = directory.replace(PROJECT_ROOT, "AI_Training")
    print(f"  OK {relative_path}/")

print("=" * 55)
print(f"Project root: {PROJECT_ROOT}")
print("All directories created (or already exist).")

---

## Cell 4 - Tao file Config

**WHAT**: File config la noi luu tat ca **hyperparameter** cua training.

**WHY**: Neu ban hard-code cac con so nay vao training code:
```python
# CACH SAI
for epoch in range(100):        # <- so 100 nam sau trong code
    optimizer = SGD(lr=0.001)   # <- learning rate nam sau trong code
```

Khi muon thu `epochs=200` hay `lr=0.01`, ban phai vao sua code -> de nham, kho so sanh.

**Cach dung**: Tat ca hyperparameter o mot cho duy nhat -> config.yaml.

**HOW**:
- `config_v1.yaml` la **template goc** - khong bao gio sua sau khi da dung de train.
- Khi bat dau experiment, **copy** config vao thu muc experiment:

```
configs/config_v1.yaml                   <- template goc (khong sua)
       |
       v copy
experiments/experiment_001/config.yaml   <- dung de train, immutable sau khi train
```

**WHEN**: Tao config truoc khi train. Moi lan thay doi setting lon -> tao config moi (config_v2.yaml).

In [ ]:
# ============================================================
# CELL 4: Create Configuration File
# Purpose: Store all hyperparameters in one place
# ============================================================

import os
import yaml

import sys
IN_COLAB = 'google.colab' in sys.modules
PROJECT_ROOT = "/content/drive/MyDrive/AI_Training" if IN_COLAB else os.path.abspath(".")

# --- Define config as a Python dictionary ---
config = {
    "seed": 42,

    "model": {
        "input_size": 1,    # 1 input feature: x
        "output_size": 1,   # 1 output: predicted y
    },

    "training": {
        "epochs": 100,
        "batch_size": 32,
        "learning_rate": 0.001,
        "early_stopping_patience": 10,
        "gradient_clip_norm": None,  # None = disabled; set e.g. 1.0 to enable
    },

    "scheduler": {
        "use_scheduler": False,          # Disabled for Phase 1 (simple model)
        "scheduler_type": "ReduceLROnPlateau",
        "scheduler_patience": 5,
        "scheduler_factor": 0.5,
    },

    "data": {
        "train_ratio": 0.70,       # 70% for training
        "validation_ratio": 0.15,  # 15% for validation
        "test_ratio": 0.15,        # 15% for test
        "dataset_version": "v1",
        "num_samples": 1000,
    },
}

# --- Save config to Google Drive ---
config_path = f"{PROJECT_ROOT}/configs/config_v1.yaml"

with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"Config saved to: {config_path}")
print()
print("Config contents:")
print("-" * 40)

# Read back and display
with open(config_path, "r") as f:
    print(f.read())

---

## Cell 5 - Random Seed (Reproducibility)

**WHAT**: Random Seed la mot con so co dinh dung de khoi tao bo sinh so ngau nhien.

**WHY**: PyTorch, NumPy dung so ngau nhien o **rat nhieu cho**:
- Khoi tao weight cua model (khong the bat dau tu 0)
- Shuffle thu tu data trong DataLoader
- Dropout (tat ngau nhien mot so neuron)

Neu khong fix seed -> moi lan chay lai cho ket qua khac nhau -> khong the so sanh hai experiment cong bang.

**Vi du thuc te**: Ban thu `lr=0.001` duoc loss=0.05. Ban doi sang `lr=0.01` duoc loss=0.03.
Nhung neu seed khac nhau, weight khoi tao khac nhau -> ban khong biet ket qua tot hon la do learning rate hay do may man.

**HOW**: Fix seed cho **tat ca** thu vien dung random:
- `torch` -- PyTorch operations
- `torch.cuda` -- GPU operations
- `numpy` -- NumPy arrays
- `random` -- Python built-in random

**WHEN**: **Ngay dau moi experiment**, truoc khi tao dataset, model, hay DataLoader.

Seed phai duoc luu trong config de co the reproduce lai ket qua chinh xac sau nay.

In [ ]:
# ============================================================
# CELL 5: Set Random Seed for Reproducibility
# Purpose: Ensure experiments produce the same results every run
# ============================================================

import torch
import numpy as np
import random
import yaml

import sys
IN_COLAB = 'google.colab' in sys.modules
PROJECT_ROOT = "/content/drive/MyDrive/AI_Training" if IN_COLAB else os.path.abspath(".")

# --- Load config ---
config_path = f"{PROJECT_ROOT}/configs/config_v1.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

SEED = config["seed"]


def set_seed(seed):
    """
    Fix random seed for all libraries to ensure reproducibility.

    Must be called at the beginning of each experiment,
    before creating any dataset, model, or DataLoader.
    """
    # PyTorch CPU operations
    torch.manual_seed(seed)

    # PyTorch GPU operations
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # For multi-GPU (safe to call even with 1 GPU)

    # NumPy operations
    np.random.seed(seed)

    # Python built-in random
    random.seed(seed)

    # Make cuDNN deterministic (slightly slower but reproducible)
    # This ensures GPU convolution operations are deterministic
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# --- Apply seed ---
set_seed(SEED)

print("=" * 55)
print(f"  Random seed set to: {SEED}")
print("=" * 55)
print(f"  torch.manual_seed({SEED})          OK")
print(f"  torch.cuda.manual_seed({SEED})     OK")
print(f"  np.random.seed({SEED})             OK")
print(f"  random.seed({SEED})                OK")
print(f"  cudnn.deterministic = True         OK")
print("=" * 55)
print("Seed applied. Results are now reproducible.")

---

## Cell 6 - Kiem tra toan bo setup

**WHAT**: Kiem tra tat ca moi thu da san sang.

**WHY**: Truoc khi bat dau Phase 2, can xac nhan:
- Google Drive da mount
- Cau truc thu muc da tao
- Config da ton tai
- Device (GPU/CPU) da xac dinh
- Seed da set

**WHEN**: Chay sau khi hoan thanh tat ca cell tren.

In [ ]:
# ============================================================
# CELL 6: Verify Complete Setup
# Purpose: Final check that everything is ready for Phase 2
# ============================================================

import os
import torch

import sys
IN_COLAB = 'google.colab' in sys.modules
PROJECT_ROOT = "/content/drive/MyDrive/AI_Training" if IN_COLAB else os.path.abspath(".")

# --- Check list ---
checks = []

# 1. Google Drive mounted
drive_mounted = os.path.exists("/content/drive/MyDrive")
checks.append(("Google Drive mounted", drive_mounted))

# 2. Project root exists
project_exists = os.path.exists(PROJECT_ROOT)
checks.append(("Project root exists", project_exists))

# 3. Config file exists
config_path = f"{PROJECT_ROOT}/configs/config_v1.yaml"
config_exists = os.path.isfile(config_path)
checks.append(("Config v1 exists", config_exists))

# 4. Key directories exist
key_dirs = [
    "data/raw/dataset_v1",
    "models/checkpoints",
    "models/final",
    "experiments/experiment_001",
    "src",
    "plots",
]
for d in key_dirs:
    full_path = f"{PROJECT_ROOT}/{d}"
    checks.append((f"Directory: {d}", os.path.exists(full_path)))

# 5. PyTorch available
checks.append(("PyTorch available", True))  # Would have failed earlier if not

# 6. Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Print results ---
print("=" * 55)
print("         SETUP VERIFICATION")
print("=" * 55)

all_passed = True
for name, passed in checks:
    status = "OK" if passed else "FAIL"
    print(f"  [{status}] {name}")
    if not passed:
        all_passed = False

print("-" * 55)
print(f"  Device : {device}")
if torch.cuda.is_available():
    print(f"  GPU    : {torch.cuda.get_device_name(0)}")
print("=" * 55)

if all_passed:
    print("ALL CHECKS PASSED -- Ready for Phase 2!")
else:
    print("Some checks failed. Re-run the cells above first.")

print("=" * 55)

---

## Tom tat Phase 1 - Environment

### Ban da hoc duoc:

| Khai niem | Giai thich ngan |
|-----------|----------------|
| **Google Colab** | May tinh di muon - data mat khi reset runtime |
| **Google Drive** | Noi luu tru vinh vien, mount vao `/content/drive/` |
| **GPU** | Hang nghin core tinh song song - ly tuong cho AI |
| **CUDA** | Nen tang NVIDIA de dung GPU trong Python |
| **PyTorch** | Thu vien AI - tu dong tinh gradient |
| **Config** | File chua tat ca hyperparameter - khong hard-code |
| **Random Seed** | Fix de moi run cho ket qua giong nhau (reproducibility) |
| **Project Structure** | To chuc thu muc ro rang - data, models, configs, experiments |

---

### Quy tac vang:

**KHONG BAO GIO** luu checkpoint quan trong o `/content/` - chi luu tren Google Drive.

**KHONG BAO GIO** hard-code hyperparameter trong training code - luon doc tu config.

**LUON LUON** set seed truoc khi tao bat ky dataset, model, hay DataLoader nao.

---

### Cau hoi kiem tra

Hay tra loi cac cau hoi sau truoc khi sang Phase 2:

**Cau 1**: Tai sao phai mount Google Drive? Dieu gi xay ra neu ban chi luu file o `/content/`?

**Cau 2**: Neu ban chay training 2 lan voi cung code nhung khong set seed, ket qua co giong nhau khong? Tai sao?

**Cau 3**: Tai sao `config_v1.yaml` la template goc khong duoc sua, con `experiments/experiment_001/config.yaml` la ban copy dung de train?

---

### Bai tap nho

1. Chay tat ca 6 cell theo thu tu.
2. Kiem tra tren Google Drive rang thu muc `AI_Training/` da duoc tao.
3. Mo file `configs/config_v1.yaml` tren Drive va doc noi dung.
4. Thu thay doi `seed: 42` thanh `seed: 123` trong Cell 5 va xem output co thay doi khong.

---

**DUNG TAI DAY.**

Khi ban da tra loi duoc 3 cau hoi tren, hay noi **"Tiep tuc"** de bat dau **Phase 2 - Tensor**.

---

# Phase 2 - Tensor

```
Phase 1  -> Environment       <- HOAN THANH
Phase 2  -> Tensor            <- BAN DANG O DAY
Phase 3  -> Dataset
Phase 4  -> Model
Phase 5  -> Training
Phase 6  -> Evaluation
Phase 7  -> Checkpoint
Phase 8  -> Stability
Phase 9  -> Experiment
Phase 10 -> Scaling
Phase 11 -> Advanced AI
```

---

## Muc tieu Phase 2

Truoc khi co Dataset, Model, hay Training Loop - ban can hieu **Tensor**.

Tensor la **ngon ngu** cua PyTorch. Moi thu trong PyTorch deu la Tensor:

```
Input data  -> Tensor
Weight      -> Tensor
Bias        -> Tensor
Prediction  -> Tensor
Loss        -> Tensor
Gradient    -> Tensor
```

Neu khong hieu Tensor, ban se bi lac khi doc bat ky training code nao.

---

Trong Phase 2, ban se hoc:

| Cell    | Noi dung |
|---------|----------|
| Cell 7  | Tensor la gi? Tao Tensor nhu the nao? |
| Cell 8  | Shape - hinh dang cua Tensor |
| Cell 9  | Dtype - kieu du lieu cua Tensor |
| Cell 10 | Device - Tensor tren CPU vs GPU |
| Cell 11 | Phep tinh co ban voi Tensor |

---

## Cell 7 - Tensor

**WHAT**: Tensor la cau truc du lieu co nhieu chieu (multi-dimensional array) cua PyTorch.

Neu ban biet NumPy, Tensor giong nhu `np.ndarray` nhung co them 2 sieu nang:
1. Chay tren **GPU** (tinh nhanh hon hang tram lan)
2. Tu dong tinh **gradient** (can cho backpropagation)

---

**WHY**: AI la bai toan tinh toan tren so. Moi thu phai duoc bieu dien bang so:

```
x = 5.0           -> Tensor([5.0])              <- 1 con so (scalar)
x = [1, 2, 3]     -> Tensor([1, 2, 3])          <- day so (vector)
Anh 28x28 pixel   -> Tensor shape [28, 28]      <- ma tran (matrix)
Video 100 khung   -> Tensor shape [100,28,28,3] <- nhieu chieu
```

**HOW**: Co 4 cach tao Tensor pho bien:

```python
# Cach 1: Tu list Python
t = torch.tensor([1.0, 2.0, 3.0])

# Cach 2: Toan mot gia tri
t = torch.zeros(3)      # [0., 0., 0.]
t = torch.ones(3)       # [1., 1., 1.]

# Cach 3: So ngau nhien
t = torch.rand(3)       # [0.23, 0.81, 0.45]  - phan phoi deu [0, 1]
t = torch.randn(3)      # [-0.5, 1.2, 0.3]   - phan phoi chuan

# Cach 4: Khoang gia tri (giong range() cua Python)
t = torch.arange(0, 5)  # [0, 1, 2, 3, 4]
```

**WHEN**: Moi khi co du lieu (so, danh sach, mang) can dua vao PyTorch de tinh toan,
ban phai chuyen no sang Tensor truoc.

In [ ]:
# ============================================================
# CELL 7: Tensor - Co ban
# Purpose: Understand what Tensor is and how to create one
# ============================================================

import torch

print('=' * 55)
print('         TENSOR - CO BAN')
print('=' * 55)

# --- 0D Tensor: Scalar (1 con so don le) ---
# Vi du: loss = 0.045  <- day la mot scalar tensor
scalar = torch.tensor(3.14)
print('\n[0D - Scalar]')
print(f'  Gia tri  : {scalar}')
print(f'  Kieu     : {type(scalar)}')
print(f'  So chieu : {scalar.ndim}')  # 0 chieu

# --- 1D Tensor: Vector (day so) ---
# Vi du: [x1, x2, x3, x4, x5] <- 5 input mau
vector = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
print('\n[1D - Vector]')
print(f'  Gia tri  : {vector}')
print(f'  So chieu : {vector.ndim}')  # 1 chieu

# --- 2D Tensor: Matrix (bang so) ---
# Vi du: batch chua 3 mau, moi mau co 4 feature
matrix = torch.tensor([
    [1.0, 2.0, 3.0, 4.0],
    [5.0, 6.0, 7.0, 8.0],
    [9.0, 10.0, 11.0, 12.0],
])
print('\n[2D - Matrix]')
print(f'  Gia tri  :\n{matrix}')
print(f'  So chieu : {matrix.ndim}')  # 2 chieu

# --- 4 cach tao Tensor pho bien ---
print('\n' + '-' * 40)
print('4 cach tao Tensor:')
print('-' * 40)

# Cach 1: Tu list Python
t1 = torch.tensor([10.0, 20.0, 30.0])
print(f'  torch.tensor([10, 20, 30])   -> {t1}')

# Cach 2: Toan so 0 hoac 1
t2 = torch.zeros(3)
t3 = torch.ones(3)
print(f'  torch.zeros(3)               -> {t2}')
print(f'  torch.ones(3)                -> {t3}')

# Cach 3: So ngau nhien
torch.manual_seed(42)  # Fix seed de output on dinh
t4 = torch.rand(3)     # Phan phoi deu [0, 1]
t5 = torch.randn(3)    # Phan phoi chuan (mean=0, std=1)
print(f'  torch.rand(3)                -> {t4.round(decimals=3)}')
print(f'  torch.randn(3)               -> {t5.round(decimals=3)}')

# Cach 4: Khoang gia tri
t6 = torch.arange(0, 5)
print(f'  torch.arange(0, 5)           -> {t6}')

print('=' * 55)
print('TENSOR la don vi co ban nhat cua PyTorch.')
print('Moi thu (data, weight, loss) deu la Tensor.')
print('=' * 55)

---

## Cell 8 - Shape

**WHAT**: `shape` la **hinh dang** cua Tensor - co bao nhieu phan tu tren moi chieu.

Vi du:
```
Tensor([5.0])         -> shape: torch.Size([1])      <- 1 phan tu
Tensor([1, 2, 3])     -> shape: torch.Size([3])      <- 3 phan tu
Tensor 3 x 4          -> shape: torch.Size([3, 4])   <- 3 hang, 4 cot
Tensor 2 x 3 x 4      -> shape: torch.Size([2,3,4])  <- 3 chieu
```

---

**WHY**: Shape la dieu ban se lam viec nhieu NHAT khi debug AI.

Loi pho bien nhat:
```
RuntimeError: mat1 and mat2 shapes cannot be multiplied ([32] and [1, 1])
```
Hieu shape giup ban giai quyet loi nay trong 10 giay thay vi 1 gio.

**HOW**: Cac cach doc va thay doi shape:

```python
t = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])  # 3 hang, 2 cot

t.shape          # torch.Size([3, 2])
t.size()         # torch.Size([3, 2])  - tuong duong
t.ndim           # 2  - so chieu
t.numel()        # 6  - tong so phan tu

# Thay doi shape (KHONG thay doi du lieu):
t.view(2, 3)     # reshape thanh 2x3
t.view(-1)       # flatten thanh 1D, -1 = PyTorch tu tinh

# Them hoac bot chieu:
t.unsqueeze(0)   # [3,2] -> [1,3,2] - them chieu o vi tri 0
t.squeeze()      # Bo tat ca chieu co kich thuoc 1
```

**WHEN**: Kiem tra shape truoc khi:
- Nap du lieu vao model
- Tinh loss
- Moi khi gap loi shape mismatch

In [ ]:
# ============================================================
# CELL 8: Shape - Hinh dang cua Tensor
# Purpose: Understand and manipulate Tensor shapes
# ============================================================

import torch

print('=' * 55)
print('         SHAPE - HINH DANG CUA TENSOR')
print('=' * 55)

# --- Tao cac Tensor voi nhieu shape khac nhau ---
t_1d = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
t_2d = torch.tensor([
    [1.0, 2.0, 3.0, 4.0],
    [5.0, 6.0, 7.0, 8.0],
    [9.0, 10.0, 11.0, 12.0],
])
t_3d = torch.ones(2, 3, 4)  # 2 blocks, 3 rows, 4 cols

# --- Xem shape ---
print('\n[Doc Shape]')
print(f'  1D tensor shape : {t_1d.shape}   <- [5 phan tu]')
print(f'  2D tensor shape : {t_2d.shape} <- [3 hang, 4 cot]')
print(f'  3D tensor shape : {t_3d.shape} <- [2 block, 3 hang, 4 cot]')

# --- ndim va numel ---
print('\n[Cac thong tin them]')
print(f'  t_2d.ndim    = {t_2d.ndim}    <- so chieu')
print(f'  t_2d.numel() = {t_2d.numel()}  <- tong so phan tu (3 x 4 = 12)')
print(f'  t_3d.numel() = {t_3d.numel()}  <- tong so phan tu (2 x 3 x 4 = 24)')

# --- Thay doi shape voi .view() ---
print('\n[Thay doi Shape voi .view()]')
print(f'  t_2d ban dau  : shape {t_2d.shape}')

reshaped_2x6 = t_2d.view(2, 6)   # 3x4 -> 2x6  (12 phan tu giu nguyen)
reshaped_1d  = t_2d.view(-1)     # 3x4 -> [12]  (flatten)
print(f'  .view(2, 6)   : shape {reshaped_2x6.shape}')
print(f'  .view(-1)     : shape {reshaped_1d.shape}   <- flatten, -1 tu tinh')

# --- unsqueeze va squeeze ---
print('\n[Them / Bot Chieu]')
t_base = torch.tensor([1.0, 2.0, 3.0])  # shape [3]
t_col  = t_base.unsqueeze(1)             # shape [3, 1]  <- them chieu cot
t_row  = t_base.unsqueeze(0)             # shape [1, 3]  <- them chieu hang
t_back = t_col.squeeze()                 # shape [3]     <- bot chieu thua

print(f'  ban dau         : {t_base.shape}')
print(f'  .unsqueeze(1)   : {t_col.shape}   <- them chieu cot')
print(f'  .unsqueeze(0)   : {t_row.shape}   <- them chieu hang')
print(f'  .squeeze()      : {t_back.shape}      <- bot chieu thua')

# --- Tai sao unsqueeze quan trong trong AI? ---
print('\n[Vi du thuc te trong bai y = 2x + 1]')
x_samples = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])  # 5 mau
print(f'  x_samples.shape   = {x_samples.shape}  <- 5 gia tri x')
x_for_model = x_samples.unsqueeze(1)                   # [5] -> [5, 1]
print(f'  x_for_model.shape = {x_for_model.shape} <- Linear layer can [batch_size, n_features]')

print('=' * 55)
print('SHAPE la thu ban se debug nhieu nhat khi lam AI.')
print('Khi gap loi: in .shape truoc, moi lam thu khac.')
print('=' * 55)

---

## Cell 9 - Dtype

**WHAT**: `dtype` la **kieu du lieu** cua Tensor.
Moi phan tu trong Tensor duoc luu bang kieu so nao?

| Dtype | Giai thich | Mac dinh cho |
|-------|------------|-------------|
| `torch.float32` | So thuc 32-bit | **Weight, Loss, Input** |
| `torch.float64` | So thuc 64-bit | Khong can trong AI |
| `torch.int64`   | So nguyen 64-bit | **Label classification** |
| `torch.bool`    | True / False | Mask, filter |

---

**WHY**: Kieu du lieu sai la nguyen nhan cua loi kho hieu nhat:

```
RuntimeError: expected scalar type Float but found Long
```

Long = `int64`. Ban dua `int64` vao model trong khi model can `float32`.

**HOW**:
```python
# Khai bao dtype khi tao:
t = torch.tensor([1.0, 2.0], dtype=torch.float32)

# Chuyen doi:
t_int   = torch.tensor([1, 2, 3])      # int64 (mac dinh cho so nguyen)
t_float = t_int.float()                # -> float32
t_float = t_int.to(torch.float32)      # Cach tuong duong, ro rang hon
```

**WHEN**: Kiem tra dtype khi:
- Tao dataset tu file hoac tu list Python
- Model bao loi dtype mismatch

**Quy tac vang**: Weight va input phai cung dtype - thuong la `float32`.

In [ ]:
# ============================================================
# CELL 9: Dtype - Kieu du lieu cua Tensor
# Purpose: Understand data types and avoid dtype mismatch errors
# ============================================================

import torch

print('=' * 55)
print('         DTYPE - KIEU DU LIEU CUA TENSOR')
print('=' * 55)

# --- Dtype mac dinh ---
print('\n[Dtype mac dinh]')

t_from_float = torch.tensor([1.0, 2.0, 3.0])
t_from_int   = torch.tensor([1, 2, 3])

print(f'  torch.tensor([1.0, 2.0]).dtype = {t_from_float.dtype}  <- so thap phan -> float32')
print(f'  torch.tensor([1, 2, 3]).dtype  = {t_from_int.dtype}    <- so nguyen    -> int64')

# --- Cac kieu dtype pho bien ---
print('\n[Cac Dtype pho bien trong AI]')

t_f32  = torch.tensor([1.0],         dtype=torch.float32)
t_f64  = torch.tensor([1.0],         dtype=torch.float64)
t_i64  = torch.tensor([1],           dtype=torch.int64)
t_bool = torch.tensor([True, False], dtype=torch.bool)

print(f'  float32 : {t_f32.dtype}  <- QUAN TRONG NHAT - weight va input')
print(f'  float64 : {t_f64.dtype}  <- it dung trong AI (ton bo nho gap doi)')
print(f'  int64   : {t_i64.dtype}    <- label cua bai toan phan loai')
print(f'  bool    : {t_bool.dtype}     <- mask, filter')

# --- Chuyen doi dtype ---
print('\n[Chuyen doi Dtype]')

t_int = torch.tensor([1, 2, 3])   # int64
print(f'  Ban dau            : {t_int}  dtype={t_int.dtype}')

t_float = t_int.float()           # -> float32
print(f'  .float()           : {t_float}  dtype={t_float.dtype}')

t_float2 = t_int.to(torch.float32)  # Cach tuong duong
print(f'  .to(torch.float32) : {t_float2}  dtype={t_float2.dtype}')

# --- Vi du loi pho bien va cach tranh ---
print('\n[Vi du: Loi Dtype trong bai y = 2x + 1]')

x_wrong = torch.tensor([1, 2, 3, 4, 5])              # int64  <- SAI cho model
x_right = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])   # float32 <- DUNG

print(f'  x_wrong dtype = {x_wrong.dtype}  <- Neu nap vao model -> loi!')
print(f'  x_right dtype = {x_right.dtype}  <- Dung cho model')

x_fixed = x_wrong.float()
print(f'  x_wrong.float()    -> dtype={x_fixed.dtype}  <- da sua')

print('=' * 55)
print('QUY TAC: Weight va Input luon phai la float32.')
print('Khi gap loi dtype -> kiem tra .dtype truoc tien.')
print('=' * 55)

---

## Cell 10 - Device

**WHAT**: `device` la **noi Tensor dang song** - tren CPU hay GPU.

```
device=cpu   <- Tensor dang tren RAM may tinh
device=cuda  <- Tensor dang tren VRAM cua GPU
```

---

**WHY**: GPU co hang nghin core tinh song song:

```
CPU  :    8 core  -> tinh  8 phep nhan cung luc
GPU  : 2560 core  -> tinh 2560 phep nhan cung luc
```

Khi train AI, ban thuc hien hang trieu phep nhan ma tran moi epoch.
GPU nhanh hon CPU **10x - 100x** cho loai tinh toan nay.

---

**HOW**:
```python
# Xac dinh device tu dong (khuyen dung)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Chuyen Tensor len device
t = torch.tensor([1.0, 2.0, 3.0])
t_on_device = t.to(device)     # linh hoat: hoat dong ca CPU lan GPU

# Chuyen Tensor ve CPU
t_back = t_on_device.cpu()
```

---

**WHEN**: Model va data **PHAI** o cung device.

Loi nay xay ra rat thuong:
```
RuntimeError: Expected all tensors to be on the same device,
but found at least two devices, cuda:0 and cpu!
```

Nguyen nhan: Model o GPU, nhung quen chuyen input len GPU.

**Giai phap**: Luon chuyen ca model VA data len device bang `.to(device)`.

In [ ]:
# ============================================================
# CELL 10: Device - CPU vs GPU
# Purpose: Understand device placement and avoid device mismatch
# ============================================================

import torch

print('=' * 55)
print('         DEVICE - CPU vs GPU')
print('=' * 55)

# --- Xac dinh device ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\n  Device hien tai: {device}')

if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
else:
    print('  Dang chay tren CPU (khong co GPU)')
    print('  Tip: Vao Runtime -> Change runtime type -> GPU de bat GPU')

# --- Tao Tensor tren CPU (mac dinh) ---
print('\n[Tao Tensor]')
t_cpu = torch.tensor([1.0, 2.0, 3.0])
print(f'  t_cpu.device = {t_cpu.device}  <- MAC DINH la CPU')

# --- Chuyen sang device ---
print('\n[Chuyen sang Device]')
t_on_device = t_cpu.to(device)
print(f'  sau .to(device): {t_on_device.device}')

if torch.cuda.is_available():
    print('  -> Tensor da duoc chuyen len GPU!')
else:
    print('  -> Tensor van o CPU (vi khong co GPU)')

# --- Loi pho bien: device mismatch ---
print('\n[Loi Pho Bien - Device Mismatch]')
print('  Neu model o cuda nhung input o cpu -> loi:')
print('  RuntimeError: Expected all tensors to be on the same device')
print()
print('  CACH TRANH:')
print('  Luon chuyen ca model VA input len cung device:')
print()
print('  model = model.to(device)   # <- chuyen model')
print('  x = x.to(device)           # <- chuyen input')
print('  y = y.to(device)           # <- chuyen label')

# --- Demo ket hop shape + dtype + device ---
print('\n[Tong Hop: Shape + Dtype + Device]')
print('  Bai toan y = 2x + 1, 5 mau du lieu:')

# Tao input dung chuan: float32, shape [5, 1], dung device
x = torch.tensor([[1.0], [2.0], [3.0], [4.0], [5.0]])   # shape [5, 1]
y = torch.tensor([[3.0], [5.0], [7.0], [9.0], [11.0]])  # shape [5, 1]

x = x.to(device)
y = y.to(device)

print(f'  x: shape={x.shape}, dtype={x.dtype}, device={x.device}')
print(f'  y: shape={y.shape}, dtype={y.dtype}, device={y.device}')
print()
print('  -> San sang de nap vao model!')

print('=' * 55)
print('DEVICE: Luon dung .to(device) cho ca model va data.')
print('KHONG BAO GIO de model o cuda nhung data o cpu.')
print('=' * 55)

---

## Cell 11 - Phep tinh co ban voi Tensor

**WHAT**: Tensor ho tro cac phep tinh so hoc co ban: cong, tru, nhan, chia.
Va quan trong hon het: **nhan ma tran**.

**WHY**: Toan bo tinh toan trong AI la phep nhan ma tran:

```
prediction = W @ x + b
```

Trong do:
- `W` = Weight (Tensor)
- `x` = Input (Tensor)
- `b` = Bias (Tensor)
- `@` = phep nhan ma tran (matrix multiplication)

**HOW**:
```python
# Phep tinh element-wise (tung cap phan tu):
a + b              # Cong
a - b              # Tru
a * b              # Nhan tung phan tu (KHONG phai nhan ma tran)
a / b              # Chia

# Phep nhan ma tran:
a @ b              # Ky hieu ngan
torch.matmul(a, b) # Ky hieu day du

# Lay Python float tu scalar Tensor:
loss_tensor.item() # PHAI dung .item() khi in so, luu CSV
```

**WHEN**: Hieu phep nhan ma tran de sau nay hieu tai sao Linear layer hoat dong.

In [ ]:
# ============================================================
# CELL 11: Phep tinh co ban voi Tensor
# Purpose: Understand basic tensor operations used in AI
# ============================================================

import torch

print('=' * 55)
print('         PHEP TINH CO BAN VOI TENSOR')
print('=' * 55)

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

# --- Phep tinh element-wise ---
print('\n[Phep tinh tung phan tu (element-wise)]')
print(f'  a          = {a}')
print(f'  b          = {b}')
print(f'  a + b      = {a + b}')
print(f'  a - b      = {a - b}')
print(f'  a * b      = {a * b}    <- KHONG phai nhan ma tran')
print(f'  a / b      = {(a / b).round(decimals=3)}')
print(f'  a ** 2     = {a ** 2}')

# --- Aggregate operations ---
print('\n[Phep tinh tong hop]')
print(f'  a.sum()    = {a.sum().item():.1f}    <- Tong tat ca phan tu')
print(f'  a.mean()   = {a.mean().item():.1f}    <- Trung binh')
print(f'  a.min()    = {a.min().item():.1f}    <- Nho nhat')
print(f'  a.max()    = {a.max().item():.1f}    <- Lon nhat')

# --- .item() - lay gia tri Python tu scalar Tensor ---
print('\n[.item() - Lay gia tri Python tu Tensor]')
loss_tensor = torch.tensor(0.045)
loss_value  = loss_tensor.item()    # Chuyen sang Python float
print(f'  loss_tensor        = {loss_tensor}   <- Tensor')
print(f'  loss_tensor.item() = {loss_value}  <- Python float')
print(f'  type()             = {type(loss_value)}')
print('  Dung .item() de in so ra man hinh, luu vao list, ghi vao CSV')

# --- Nhan ma tran ---
print('\n[Nhan Ma Tran - Trai tim cua AI]')
print('  Bai toan: y = W @ x + b')
print('  W = [[2.0]], x = [[5.0]], b = [[1.0]]')

W = torch.tensor([[2.0]])   # Weight: shape [1, 1]
x = torch.tensor([[5.0]])   # Input:  shape [1, 1]
b = torch.tensor([[1.0]])   # Bias:   shape [1, 1]

prediction = torch.matmul(W, x) + b   # equivalently: W @ x + b
print()
print(f'  W @ x + b = {W.item()} * {x.item()} + {b.item()} = {prediction.item()}')
print(f'  => Du doan y = {prediction.item()}')
print(f'  => Dung! Voi x=5: y = 2*5 + 1 = 11')

# --- Batch: Nhieu mau cung luc ---
print('\n[Batch: Tinh nhieu mau cung luc]')
print('  Day chinh la suc manh cua GPU!')

x_batch = torch.tensor([[1.0], [2.0], [3.0], [4.0], [5.0]])  # shape [5, 1]
W_batch = torch.tensor([[2.0]])   # shape [1, 1]
b_batch = torch.tensor([[1.0]])   # shape [1, 1]

# PyTorch tu dong broadcast: tinh cho ca 5 mau mot luc
y_pred = x_batch @ W_batch.T + b_batch
print(f'  x_batch: {x_batch.squeeze().tolist()}  (shape {x_batch.shape})')
print(f'  y_pred : {y_pred.squeeze().tolist()}  (shape {y_pred.shape})')
print('  => 5 du doan cho 5 mau, chi mot phep nhan ma tran!')

print('=' * 55)
print('Tensor + phep nhan ma tran = nen tang cua moi AI model.')
print('=' * 55)

---

## Tom tat Phase 2 - Tensor

### Ban da hoc duoc:

| Khai niem | Giai thich ngan |
|-----------|----------------|
| **Tensor** | Cau truc du lieu co nhieu chieu cua PyTorch. Moi thu trong AI deu la Tensor |
| **Shape** | Hinh dang Tensor: so phan tu tren moi chieu. Loi shape = loi pho bien nhat |
| **Dtype** | Kieu du lieu: `float32` cho weight/input, `int64` cho label |
| **Device** | Noi Tensor song: CPU hay GPU. Model va data PHAI cung device |
| **.item()** | Lay Python float tu scalar Tensor, dung de in so va luu metrics |
| **matmul (@)** | Nhan ma tran - trai tim cua moi AI model |

---

### Luong du lieu trong bai toan y = 2x + 1:

```
x = 5.0
    |
    v
x_tensor = torch.tensor([[5.0]])   shape=[1,1], dtype=float32, device=cpu
    |
    v
x_tensor.to(device)                chuyen len GPU neu co
    |
    v
prediction = W @ x + b             phep nhan ma tran
    |
    v
prediction = tensor([[11.0]])      shape=[1,1], dtype=float32
    |
    v
prediction.item() = 11.0           Python float - dung de in, luu
```

---

### 3 quy tac de nho:

**Quy tac 1**: Khi gap loi -> in `.shape` truoc tien.

**Quy tac 2**: Input cho model phai la `float32`.

**Quy tac 3**: Model va data phai cung device. Luon dung `.to(device)`.

---

### Cau hoi kiem tra

**Cau 1**: `torch.tensor([1, 2, 3]).dtype` tra ve gi? Tai sao day la van de khi dua vao Linear model?

**Cau 2**: Tensor co shape `[5]` va shape `[5, 1]` khac nhau o diem gi? Cai nao dung cho Linear layer?

**Cau 3**: Neu model dang o `device=cuda` nhung ban dua vao input `device=cpu`, loi gi xay ra? Cach sua?

---

### Bai tap nho

1. Tao Tensor tu list `[10, 20, 30, 40, 50]` - kiem tra dtype.
2. Chuyen Tensor do sang `float32`.
3. Reshape no thanh shape `[5, 1]`.
4. Chuyen no len device phu hop.
5. In ra shape, dtype, device cua Tensor sau moi buoc.

---

**DUNG TAI DAY.**

Khi ban da tra loi duoc 3 cau hoi tren, hay noi **"Tiep tuc"** de bat dau **Phase 3 - Dataset**.

---

# Phase 3 - Dataset

```
Phase 1  -> Environment       <- HOAN THANH
Phase 2  -> Tensor            <- HOAN THANH
Phase 3  -> Dataset           <- BAN DANG O DAY
Phase 4  -> Model
Phase 5  -> Training
Phase 6  -> Evaluation
Phase 7  -> Checkpoint
Phase 8  -> Stability
Phase 9  -> Experiment
Phase 10 -> Scaling
Phase 11 -> Advanced AI
```

---

## Muc tieu Phase 3

Ban da hieu **Tensor** - don vi co ban cua PyTorch.

Bay gio ban can hieu **Dataset** - noi du lieu duoc tao ra va to chuc.

Trong Phase 3, ban se hoc:

| Cell    | Noi dung |
|---------|----------|
| Cell 12 | Generate dataset: tao 1,000 mau du lieu cho y = 2x + 1 |
| Cell 13 | Dataset Versioning: luu dataset len Google Drive |
| Cell 14 | Train / Validation / Test Split: chia du lieu |
| Cell 15 | Kiem tra dataset: xac nhan du lieu san sang cho training |

---

### Tai sao can Dataset?

```
AI hoc tu du lieu.

Khong co du lieu -> Khong the train.
Du lieu sai     -> Model hoc sai.
Du lieu it      -> Model khong tong quat hoa duoc.
```

Bai toan cua chung ta:

```
y = 2x + 1
```

Model phai **hoc** quy luat nay tu du lieu - khong biet truoc cong thuc.

---

## Cell 12 - Generate Dataset

**WHAT**: Tao 1,000 cap du lieu `(x, y)` theo cong thuc `y = 2x + 1` co them **nhieu** (noise).

---

**WHY**: Tai sao can them noise?

Du lieu thuc te **khong bao gio** la cong thuc toan hoc ly tuong.

Vi du trong thuc te:

```
Gia nha     = 2 * dien_tich + 1  (trieu dong/m2)

Nhung thuc te:
  Nha 50m2  -> 101 trieu  (dung: 2*50+1=101)
  Nha 50m2  -> 105 trieu  (chenh lech: vi tri dep hon)
  Nha 50m2  ->  98 trieu  (chenh lech: huong bat loi)
```

Noise = nhung chenh lech nho do cac yeu to khac tac dong.

Neu dataset qua sach (khong co noise), model hoc qua tot tren train data nhung kem tren data moi -> **overfitting**.

---

**HOW**: Cong thuc tao du lieu:

```python
# x: 1000 gia tri cach deu trong khoang [-10, 10]
x = torch.linspace(-10, 10, 1000)

# noise: nhieu Gaussian nho
noise = torch.randn(1000) * 0.5   # mean=0, std=0.5

# y = 2x + 1 + noise
y = 2 * x + 1 + noise
```

**WHEN**: Tao dataset **mot lan** va luu lai. Lan sau load tu file thay vi tao lai.

---

### Giai thich: torch.linspace vs torch.rand

```
torch.linspace(-10, 10, 1000)
= Tao 1000 gia tri cach deu nhau tu -10 den 10
= [-10.0, -9.98, -9.96, ..., 9.98, 10.0]
= Trai deu tren toan bo khoang -> model hoc tot hon

torch.rand(1000) * 20 - 10
= Tao 1000 gia tri NGAU NHIEN trong khoang [-10, 10]
= Phu hop hon khi mo phong du lieu thuc te
```

In [ ]:
# ============================================================
# CELL 12: Generate Dataset
# Purpose: Create 1,000 samples for y = 2x + 1 with noise
# ============================================================

import torch
import numpy as np
import random
import yaml
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend for Colab compatibility
import matplotlib.pyplot as plt

import sys
IN_COLAB = 'google.colab' in sys.modules
PROJECT_ROOT = "/content/drive/MyDrive/AI_Training" if IN_COLAB else os.path.abspath(".")

# --- Load config ---
config_path = f"{PROJECT_ROOT}/configs/config_v1.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# --- Set seed FIRST (before any random operation) ---
SEED = config["seed"]
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

NUM_SAMPLES = config["data"]["num_samples"]  # 1000

print("=" * 55)
print("         GENERATE DATASET v1")
print("=" * 55)
print(f"Formula   : y = 2x + 1 + noise")
print(f"Samples   : {NUM_SAMPLES}")
print(f"x range   : [-10, 10]")
print(f"Noise     : Gaussian(mean=0, std=0.5)")
print(f"Seed      : {SEED}")
print("-" * 55)

# --- Generate x: evenly spaced in [-10, 10] ---
# torch.linspace(-10, 10, N) = N values equally spaced from -10 to 10
x = torch.linspace(-10, 10, NUM_SAMPLES)  # shape: [1000]

# --- Generate noise: small random disturbance ---
# torch.randn(N) = N values from standard normal distribution (mean=0, std=1)
# Multiply by 0.5 to reduce noise magnitude
noise = torch.randn(NUM_SAMPLES) * 0.5  # shape: [1000]

# --- Generate y: the target formula ---
# True relationship: y = 2x + 1
# With noise:       y = 2x + 1 + noise
y = 2 * x + 1 + noise  # shape: [1000]

# --- Reshape for Linear model: [N] -> [N, 1] ---
# Linear layer requires shape [batch_size, n_features]
# n_features = 1 (only x)
x = x.unsqueeze(1)  # [1000] -> [1000, 1]
y = y.unsqueeze(1)  # [1000] -> [1000, 1]

print(f"x shape   : {x.shape}  <- [num_samples, 1_feature]")
print(f"y shape   : {y.shape}  <- [num_samples, 1_output]")
print(f"x dtype   : {x.dtype}")
print(f"y dtype   : {y.dtype}")

# --- Preview first 5 samples ---
print()
print("First 5 samples:")
print(f"  {'x':>10}  {'y (noisy)':>12}  {'y (true)':>10}")
print(f"  {'-'*10}  {'-'*12}  {'-'*10}")
for i in range(5):
    x_val  = x[i].item()
    y_val  = y[i].item()
    y_true = 2 * x_val + 1
    print(f"  {x_val:>10.4f}  {y_val:>12.4f}  {y_true:>10.4f}")

# --- Plot dataset ---
fig, ax = plt.subplots(figsize=(8, 5))

x_plot = x.squeeze().numpy()
y_plot = y.squeeze().numpy()
y_true_plot = 2 * x_plot + 1

ax.scatter(x_plot, y_plot, s=4, alpha=0.4, color='steelblue', label='Data (with noise)')
ax.plot(x_plot, y_true_plot, color='red', linewidth=2, label='True: y = 2x + 1')

ax.set_title('Dataset v1 - y = 2x + 1 (with Gaussian noise)', fontsize=13)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend()
ax.grid(True, alpha=0.3)

plot_path = f"{PROJECT_ROOT}/plots/dataset_v1.png"
plt.tight_layout()
plt.savefig(plot_path, dpi=100)
plt.show()

print("=" * 55)
print(f"Dataset plot saved to: {plot_path}")
print("Dataset generated successfully.")
print("=" * 55)

---

## Cell 13 - Dataset Versioning

**WHAT**: Luu dataset thanh file `.pt` co version vao Google Drive.

---

**WHY**: Tai sao can versioning cho dataset?

```
Gia su ban train model_v1 voi dataset_v1 (1,000 mau).
Sau do ban them du lieu -> dataset_v2 (10,000 mau).

Cau hoi: Model_v1 tot hon hay model_v2?

Neu ban khong giu dataset_v1 rieng biet:
-> Ban khong biet ket qua do dataset hay do model.
-> Ban khong the reproduce lai ket qua cu.
-> Ban khong biet chinh xac model nao duoc train bang data nao.
```

Quy tac: **Khong bao gio sua dataset cu**. Them data moi -> tao dataset moi.

---

**HOW**: Luu x va y vao file `.pt` (PyTorch tensor file):

```python
# Luu:
torch.save({"x": x, "y": y}, path)

# Load lai:
data = torch.load(path)
x = data["x"]
y = data["y"]
```

---

**WHEN**: Luu dataset ngay sau khi generate. Load tu file moi khi can - khong generate lai.

---

### So sanh cac format luu:

| Format | Uu diem | Nhuoc diem |
|--------|---------|------------|
| `.pt`  | Nhanh, giu nguyen Tensor | Chi doc duoc bang PyTorch |
| `.csv` | Moi nguoi doc duoc | Cham, mat dtype info |
| `.npy` | Nhanh cho NumPy | Phai chuyen sang Tensor |

Trong project nay ta dung `.pt` vi dang lam viec hoan toan voi PyTorch.

In [ ]:
# ============================================================
# CELL 13: Dataset Versioning
# Purpose: Save dataset to Google Drive with version tracking
# ============================================================

import torch
import yaml
import json
import os
from datetime import datetime

import sys
IN_COLAB = 'google.colab' in sys.modules
PROJECT_ROOT = "/content/drive/MyDrive/AI_Training" if IN_COLAB else os.path.abspath(".")

# --- Load config ---
config_path = f"{PROJECT_ROOT}/configs/config_v1.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

DATASET_VERSION = config["data"]["dataset_version"]  # "v1"
DATASET_DIR = f"{PROJECT_ROOT}/data/raw/dataset_{DATASET_VERSION}"

print("=" * 55)
print("         DATASET VERSIONING")
print("=" * 55)

# --- Save tensors as .pt file ---
# x and y are from Cell 12 (must run Cell 12 first)
dataset_path = f"{DATASET_DIR}/dataset.pt"

torch.save({
    "x": x,
    "y": y,
}, dataset_path)

print(f"Dataset saved to: {dataset_path}")
print(f"  x shape: {x.shape}")
print(f"  y shape: {y.shape}")

# --- Save dataset metadata (JSON) ---
# Record what this dataset contains - essential for reproducibility
metadata = {
    "version": DATASET_VERSION,
    "formula": "y = 2x + 1 + noise",
    "num_samples": x.shape[0],
    "x_range": [-10, 10],
    "noise_std": 0.5,
    "seed": config["seed"],
    "x_dtype": str(x.dtype),
    "y_dtype": str(y.dtype),
    "created_at": datetime.now().isoformat(),
}

metadata_path = f"{DATASET_DIR}/metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved to: {metadata_path}")
print()
print("Metadata contents:")
print("-" * 40)
for key, val in metadata.items():
    print(f"  {key:<15}: {val}")

# --- Verify: reload and check ---
print()
print("Verifying saved file...")
loaded = torch.load(dataset_path)
x_check = loaded["x"]
y_check = loaded["y"]
print(f"  Loaded x shape: {x_check.shape}  OK")
print(f"  Loaded y shape: {y_check.shape}  OK")
print(f"  x[0] = {x_check[0].item():.4f}, y[0] = {y_check[0].item():.4f}")

print("=" * 55)
print(f"Dataset v{DATASET_VERSION} saved and verified successfully.")
print("=" * 55)

---

## Cell 14 - Train / Validation / Test Split

**WHAT**: Chia 1,000 mau du lieu thanh 3 phan:

```
Train      : 700 mau (70%)  <- De train model
Validation : 150 mau (15%)  <- De theo doi trong qua trinh train
Test       : 150 mau (15%)  <- De danh gia cuoi cung (model chua nhin thay)
```

---

**WHY**: Tai sao phai chia 3 phan?

```
Train set:       Model HOC tu day.
                 -> Model dieu chinh weight de giam loss tren tap nay.

Validation set:  Model DUOC KIEM TRA trong qua trinh train.
                 -> Dung de phat hien overfitting.
                 -> Dung de chon best model (model co val_loss tot nhat).
                 -> KHONG dung de cap nhat weight.

Test set:        Model DUOC DANH GIA mot lan duy nhat sau khi train xong.
                 -> Mo phong du lieu moi ma model chua bao gio nhin thay.
                 -> Ket qua nay moi la ket qua that su.
                 -> KHONG dung de train hay chon hyperparameter.
```

---

**Vi du thuc te**:

```
Hoc sinh on thi:

Train set      = Sach giao khoa (hoc tu day)
Validation set = De on tap (thu suc, hoc lai, thu lai)
Test set       = De thi chinh thuc (lam mot lan, khong biet truoc)
```

---

**HOW**: Dung `torch.randperm` de xao tron truoc khi chia:

```python
# Tao mang chi so ngau nhien: [423, 7, 891, 12, ...]
indices = torch.randperm(N)

# Chia chi so
train_idx = indices[:700]
val_idx   = indices[700:850]
test_idx  = indices[850:]

# Lay du lieu theo chi so
x_train = x[train_idx]
x_val   = x[val_idx]
x_test  = x[test_idx]
```

**WHEN**: Chia du lieu **mot lan** truoc khi bat dau train. Luu ket qua de dam bao nhat quan.

In [ ]:
# ============================================================
# CELL 14: Train / Validation / Test Split
# Purpose: Split dataset into train, validation, and test sets
# ============================================================

import torch
import yaml
import os

import sys
IN_COLAB = 'google.colab' in sys.modules
PROJECT_ROOT = "/content/drive/MyDrive/AI_Training" if IN_COLAB else os.path.abspath(".")

# --- Load config ---
config_path = f"{PROJECT_ROOT}/configs/config_v1.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# --- Load dataset from Google Drive ---
DATASET_VERSION = config["data"]["dataset_version"]  # "v1"
dataset_path = f"{PROJECT_ROOT}/data/raw/dataset_{DATASET_VERSION}/dataset.pt"

loaded = torch.load(dataset_path)
x = loaded["x"]
y = loaded["y"]

N = x.shape[0]  # Total number of samples (1000)

print("=" * 55)
print("         TRAIN / VALIDATION / TEST SPLIT")
print("=" * 55)
print(f"Total samples : {N}")

# --- Calculate split sizes from config ratios ---
train_ratio = config["data"]["train_ratio"]       # 0.70
val_ratio   = config["data"]["validation_ratio"]  # 0.15
test_ratio  = config["data"]["test_ratio"]        # 0.15

n_train = int(N * train_ratio)          # 700
n_val   = int(N * val_ratio)            # 150
n_test  = N - n_train - n_val           # 150 (remainder)

print(f"Train         : {n_train} samples ({train_ratio*100:.0f}%)")
print(f"Validation    : {n_val}  samples ({val_ratio*100:.0f}%)")
print(f"Test          : {n_test}  samples ({test_ratio*100:.0f}%)")
print("-" * 55)

# --- Shuffle indices with fixed seed for reproducibility ---
# torch.randperm(N) = array of N indices in random order
# Example: [423, 7, 891, 12, 674, ...]
torch.manual_seed(config["seed"])
indices = torch.randperm(N)

# --- Split indices ---
train_idx = indices[:n_train]
val_idx   = indices[n_train : n_train + n_val]
test_idx  = indices[n_train + n_val :]

# --- Slice dataset by indices ---
x_train = x[train_idx]  # shape: [700, 1]
y_train = y[train_idx]  # shape: [700, 1]

x_val   = x[val_idx]    # shape: [150, 1]
y_val   = y[val_idx]    # shape: [150, 1]

x_test  = x[test_idx]   # shape: [150, 1]
y_test  = y[test_idx]   # shape: [150, 1]

print(f"x_train shape : {x_train.shape}")
print(f"x_val   shape : {x_val.shape}")
print(f"x_test  shape : {x_test.shape}")

# --- Save processed splits to Google Drive ---
PROCESSED_DIR = f"{PROJECT_ROOT}/data/processed"

torch.save({"x": x_train, "y": y_train}, f"{PROCESSED_DIR}/train/split.pt")
torch.save({"x": x_val,   "y": y_val  }, f"{PROCESSED_DIR}/validation/split.pt")
torch.save({"x": x_test,  "y": y_test }, f"{PROCESSED_DIR}/test/split.pt")

print()
print("Split files saved:")
print(f"  {PROCESSED_DIR}/train/split.pt")
print(f"  {PROCESSED_DIR}/validation/split.pt")
print(f"  {PROCESSED_DIR}/test/split.pt")

print("=" * 55)
print("Split complete. Data is ready for training.")
print("=" * 55)

---

## Cell 15 - Kiem tra Dataset

**WHAT**: Kiem tra toan bo dataset truoc khi sang Phase 4 - Model.

**WHY**: Mot nguyen tac quan trong trong ML:

```
Garbage In, Garbage Out
```

Du lieu sai -> Model sai -> Ket qua sai.

Phai xac nhan du lieu truoc khi train - khong nen tin tuong mo.

**HOW**: Kiem tra nhung dieu sau:

```
1. Shape dung khong?
2. Dtype dung khong? (float32)
3. Gia tri x co nam trong khoang mong muon?
4. Gia tri y co gan dung voi y = 2x + 1?
5. Tong so mau = train + val + test?
6. Co NaN hay Inf trong du lieu khong?
```

**WHEN**: Chay cell nay moi khi load dataset moi. Neu co bat ky loi nao -> debug truoc khi train.

In [ ]:
# ============================================================
# CELL 15: Verify Dataset
# Purpose: Validate dataset before training
# ============================================================

import torch
import yaml
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import sys
IN_COLAB = 'google.colab' in sys.modules
PROJECT_ROOT = "/content/drive/MyDrive/AI_Training" if IN_COLAB else os.path.abspath(".")
PROCESSED_DIR = f"{PROJECT_ROOT}/data/processed"

# --- Load splits ---
train_data = torch.load(f"{PROCESSED_DIR}/train/split.pt")
val_data   = torch.load(f"{PROCESSED_DIR}/validation/split.pt")
test_data  = torch.load(f"{PROCESSED_DIR}/test/split.pt")

x_train, y_train = train_data["x"], train_data["y"]
x_val,   y_val   = val_data["x"],   val_data["y"]
x_test,  y_test  = test_data["x"],  test_data["y"]

print("=" * 55)
print("         DATASET VERIFICATION")
print("=" * 55)

checks = []

# --- Check 1: Shape ---
check_shapes = (
    x_train.shape == torch.Size([700, 1]) and
    x_val.shape   == torch.Size([150, 1]) and
    x_test.shape  == torch.Size([150, 1])
)
checks.append(("Shape [N,1] correct", check_shapes))

# --- Check 2: Dtype float32 ---
check_dtype = (
    x_train.dtype == torch.float32 and
    y_train.dtype == torch.float32
)
checks.append(("Dtype is float32", check_dtype))

# --- Check 3: No NaN or Inf ---
no_nan = (
    not torch.isnan(x_train).any() and
    not torch.isnan(y_train).any() and
    not torch.isinf(x_train).any() and
    not torch.isinf(y_train).any()
)
checks.append(("No NaN / Inf in train", no_nan))

# --- Check 4: Total sample count ---
total = x_train.shape[0] + x_val.shape[0] + x_test.shape[0]
check_total = (total == 1000)
checks.append((f"Total = {total} (expected 1000)", check_total))

# --- Check 5: y ~ 2x + 1 (within noise tolerance) ---
y_expected   = 2 * x_train + 1
residuals    = (y_train - y_expected).abs()
max_residual = residuals.max().item()
check_formula = (max_residual < 3.0)  # 3 * std(noise=0.5) = 1.5, conservative bound
checks.append((f"Max residual |y-(2x+1)| = {max_residual:.3f} < 3.0", check_formula))

# --- Print results ---
all_passed = True
for name, passed in checks:
    status = "OK" if passed else "FAIL"
    print(f"  [{status}] {name}")
    if not passed:
        all_passed = False

print("-" * 55)

# --- Statistics ---
print("\nDataset Statistics (train set):")
print(f"  x_train  |  min: {x_train.min().item():7.3f}  max: {x_train.max().item():7.3f}  mean: {x_train.mean().item():7.3f}")
print(f"  y_train  |  min: {y_train.min().item():7.3f}  max: {y_train.max().item():7.3f}  mean: {y_train.mean().item():7.3f}")

# --- Visualization: train/val/test on one plot ---
fig, ax = plt.subplots(figsize=(9, 5))

ax.scatter(x_train.numpy(), y_train.numpy(), s=4, alpha=0.5, color='steelblue', label=f'Train ({x_train.shape[0]})')
ax.scatter(x_val.numpy(),   y_val.numpy(),   s=8, alpha=0.7, color='orange',    label=f'Validation ({x_val.shape[0]})')
ax.scatter(x_test.numpy(),  y_test.numpy(),  s=8, alpha=0.7, color='green',     label=f'Test ({x_test.shape[0]})')

x_line = torch.linspace(-10, 10, 100).numpy()
ax.plot(x_line, 2 * x_line + 1, color='red', linewidth=2, label='True: y = 2x + 1')

ax.set_title('Dataset Split: Train / Validation / Test', fontsize=13)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend()
ax.grid(True, alpha=0.3)

split_plot_path = f"{PROJECT_ROOT}/plots/dataset_v1_split.png"
plt.tight_layout()
plt.savefig(split_plot_path, dpi=100)
plt.show()

print(f"\nSplit plot saved to: {split_plot_path}")
print("=" * 55)
if all_passed:
    print("ALL CHECKS PASSED -- Dataset is ready for Phase 4!")
else:
    print("Some checks FAILED. Fix before training.")
print("=" * 55)

---

## Tom tat Phase 3 - Dataset

### Ban da hoc duoc:

| Khai niem | Giai thich ngan |
|-----------|----------------|
| **Generate Dataset** | Tao du lieu y = 2x + 1 + noise. Noise giup model tong quat hoa tot hon |
| **Dataset Versioning** | Luu dataset theo phien ban. Khong sua dataset cu, chi them dataset moi |
| **Metadata** | Luu thong tin dataset (so mau, seed, cong thuc) de biet chinh xac du lieu la gi |
| **Train split** | 70% du lieu. Model HOC tu day |
| **Validation split** | 15% du lieu. Theo doi trong qua trinh train. Phat hien overfitting |
| **Test split** | 15% du lieu. Danh gia CUOI CUNG. Chi dung mot lan sau khi train xong |
| **Verification** | Kiem tra shape, dtype, NaN, Inf truoc khi train. Garbage In -> Garbage Out |

---

### Luong du lieu da xay dung:

```
Generate 1000 mau (x, y = 2x+1+noise)
    |
    v
Luu vao data/raw/dataset_v1/dataset.pt  (+ metadata.json)
    |
    v
Xao tron ngau nhien (randperm voi seed co dinh)
    |
    v
Chia 70 / 15 / 15
    |
    +-- data/processed/train/split.pt       700 mau
    +-- data/processed/validation/split.pt  150 mau
    +-- data/processed/test/split.pt        150 mau
    |
    v
Kiem tra: shape, dtype, NaN, cong thuc
    |
    v
ALL CHECKS PASSED -- San sang cho Phase 4!
```

---

### Cau hoi kiem tra

**Cau 1**: Tai sao them noise vao du lieu khi tao dataset? Dieu gi xay ra neu dataset qua sach (khong co noise)?

**Cau 2**: Test set chi duoc dung **mot lan duy nhat** sau khi train xong. Tai sao? Dieu gi xay ra neu ban dung test set nhieu lan de chon model?

**Cau 3**: Tai sao phai luu `metadata.json` kem theo dataset? Thong tin nao trong metadata quan trong nhat?

---

### Bai tap nho

1. Chay tat ca 4 cell (Cell 12 den Cell 15) theo thu tu.
2. Mo Google Drive va kiem tra cac file sau da duoc tao:
   - `data/raw/dataset_v1/dataset.pt`
   - `data/raw/dataset_v1/metadata.json`
   - `data/processed/train/split.pt`
   - `data/processed/validation/split.pt`
   - `data/processed/test/split.pt`
3. Doc noi dung file `metadata.json` - kiem tra xem no luu nhung thong tin gi.

---

**DUNG TAI DAY.**

Khi ban da tra loi duoc 3 cau hoi tren, hay noi **"Tiep tuc"** de bat dau **Phase 4 - Model**.

---

# Phase 4 — Model

**Mục tiêu của Phase 4**:
- Hiểu kiến trúc Model trong PyTorch với `torch.nn.Module`
- Hiểu bản chất của **Linear Model** (Hồi quy tuyến tính)
- Hiểu **Weight** ($w$) và **Bias** ($b$) là gì, được khởi tạo ra sao
- Hiểu luồng tính toán **Forward Pass**: $x \rightarrow \hat{y} = w \cdot x + b$
- Thực hiện Forward pass trên dữ liệu mẫu và kiểm tra shape đầu ra

```
Input Tensor [N, 1]
       │
       ▼
┌─────────────────────────┐
│   LinearRegressionModel │
│                         │
│   Weight (w): shape [1,1]│
│   Bias   (b): shape [1] │
│                         │
│   y_pred = x * w + b    │
└─────────────────────────┘
       │
       ▼
Output Prediction [N, 1]
```

---

## Cell 16 - Linear Model & `torch.nn.Module`

**WHAT**:
- `torch.nn.Module`: Class cha (base class) cho tất cả các mô hình mạng nơ-ron trong PyTorch.
- `torch.nn.Linear(in_features, out_features)`: Lớp biến đổi tuyến tính thực hiện phép toán $\hat{y} = x \cdot W^T + b$.

**WHY**:
- Khi kế thừa từ `nn.Module`, PyTorch tự động quản lý các tham số học được (`parameters()`), theo dõi gradient, chuyển đổi thiết bị (CPU/GPU với `.to(device)`), và lưu/load checkpoint dễ dàng với `state_dict()`.
- Lớp `nn.Linear` bao bọc sẵn ma trận trọng số và vector bias giúp ta không cần phải tự tạo và quản lý tensor thủ công.

**HOW**:
1. Định nghĩa class kế thừa `nn.Module`.
2. Trong `__init__()`, gọi `super().__init__()` và khởi tạo các layer (ở đây là `nn.Linear(1, 1)` vì input $x$ có 1 chiều và output $y$ có 1 chiều).
3. Trong `forward(self, x)`, định nghĩa luồng dữ liệu truyền qua các layer.

**WHEN**:
- Luôn dùng `nn.Module` khi xây dựng bất kỳ mô hình AI nào trong PyTorch (từ Linear Regression đến Deep Neural Network hay Transformer).

In [ ]:
# ============================================================
# CELL 16: Define Linear Regression Model
# Purpose: Create a PyTorch nn.Module for y = w*x + b
# ============================================================

import torch
import torch.nn as nn

# --- Define Model Architecture ---
class LinearRegressionModel(nn.Module):
    def __init__(self, input_size: int = 1, output_size: int = 1):
        """
        Linear Regression Model: y_pred = x * weight + bias
        """
        super(LinearRegressionModel, self).__init__()
        # nn.Linear(in_features, out_features) contains trainable weight and bias
        self.linear = nn.Linear(in_features=input_size, out_features=output_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Defines the computation performed at every forward pass.
        """
        return self.linear(x)

print("=" * 55)
print("         MODEL DEFINITION")
print("=" * 55)

# Instantiate model with fixed seed for reproducibility
torch.manual_seed(42)
model = LinearRegressionModel(input_size=1, output_size=1)

print(f"Model Architecture:\n{model}")
print("=" * 55)

---

## Cell 17 - Weight và Bias trong Model

**WHAT**:
- **Weight ($w$)**: Trọng số / hệ số góc nhân với input $x$. Quyết định mức độ ảnh hưởng của $x$ lên $y$.
- **Bias ($b$)**: Độ lệch / hệ số tự do độc lập với $x$. Cho phép đường thẳng dịch chuyển lên/xuống dọc theo trục tung.

```
Công thức chuẩn: y_pred = x * w + b
Công thức gốc:   y_true = x * 2 + 1   (Target lý tưởng: w -> 2.0, b -> 1.0)
```

**WHY**:
- Nếu **không có Weight**: Model không thể co giãn hay đảo chiều quan hệ giữa $x$ và $y$.
- Nếu **không có Bias**: Đường thẳng dự đoán luôn bắt buộc phải đi qua gốc toạ độ $(0, 0)$. Khi $x=0$, $y$ luôn bằng $0$, không thể học được hàm $y = 2x + 1$ (vì khi $x=0$, $y=1$).

**HOW**:
- Khi khởi tạo, PyTorch gán giá trị ngẫu nhiên cho Weight và Bias (theo phân phối Uniform Kaiming).
- Truy cập Weight: `model.linear.weight` (Shape: `[1, 1]`).
- Truy cập Bias: `model.linear.bias` (Shape: `[1]`).
- Danh sách tham số học được: `list(model.parameters())` hoặc `model.state_dict()`.

**WHEN**:
- Weight và Bias ban đầu là ngẫu nhiên (chưa học). Trong Phase 5 (Training), Optimizer sẽ điều chỉnh dần dần hai giá trị này qua từng epoch để tiệm cận $w \approx 2.0$ và $b \approx 1.0$.

In [ ]:
# ============================================================
# CELL 17: Inspect Weight and Bias
# Purpose: Examine initial model parameters before training
# ============================================================

print("=" * 55)
print("         INITIAL PARAMETERS INSPECTION")
print("=" * 55)

# --- Access parameters via named_parameters() ---
for name, param in model.named_parameters():
    print(f"Parameter: {name:<20} | Shape: {str(param.shape):<15} | Requires Grad: {param.requires_grad}")
    print(f"  Initial Values: {param.data.tolist()}")

print("-" * 55)

# --- Extract current w and b values ---
w_init = model.linear.weight.item()
b_init = model.linear.bias.item()

print(f"Current Model Formula : y_pred = {w_init:.4f} * x + ({b_init:.4f})")
print(f"Target True Formula   : y_true = 2.0000 * x + 1.0000")
print("-" * 55)
print(f"Weight Error (|w - 2|): {abs(w_init - 2.0):.4f}")
print(f"Bias Error   (|b - 1|): {abs(b_init - 1.0):.4f}")
print("=" * 55)
print("Notice: Initial parameters are random. Training in Phase 5 will optimize them.")

---

## Cell 18 - Forward Pass (Lan truyền xuôi)

**WHAT**:
- Forward Pass là quá trình đưa input tensor $x$ qua model để tính ra kết quả dự đoán $\hat{y}$ (`y_pred`).

**WHY**:
- Cần có $\hat{y}$ để so sánh với nhãn thật $y$ và tính sai số (Loss). Từ sai số đó mới có thể tính đạo hàm (Gradient) để tối ưu model.

**HOW**:
- Gọi `y_pred = model(x)` (hoặc `model(x_batch)`).
- **Lưu ý cực kỳ quan trọng**: Luôn gọi `model(x)`, **KHÔNG** gọi trực tiếp `model.forward(x)`. PyTorch thực hiện quản lý hook, trace và profiling tự động thông qua phương thức `__call__` của class `nn.Module`.

**WHEN**:
- Chạy Forward Pass trong mỗi bước training batch, validation batch và khi dự đoán trên dữ liệu mới (Inference).

In [ ]:
# ============================================================
# CELL 18: Test Forward Pass on Sample Inputs
# Purpose: Pass test inputs through model and inspect predictions
# ============================================================

print("=" * 55)
print("         FORWARD PASS ON SAMPLE DATA")
print("=" * 55)

# --- Create sample inputs: [batch_size, 1] ---
sample_x = torch.tensor([[1.0], [2.0], [3.0], [10.0], [20.0]], dtype=torch.float32)

# --- Forward pass through model ---
# When evaluating without training, use torch.no_grad()
with torch.no_grad():
    sample_y_pred = model(sample_x)

# --- Compare Untrained Prediction vs Ground Truth ---
print(f"Input shape : {sample_x.shape}")
print(f"Output shape: {sample_y_pred.shape}")
print("-" * 55)
print(f"{'Input (x)':<10} | {'Untrained Pred (y_hat)':<22} | {'True Value (y=2x+1)':<20} | {'Diff':<10}")
print("-" * 55)

for x_val, pred_val in zip(sample_x, sample_y_pred):
    x_i = x_val.item()
    pred_i = pred_val.item()
    true_i = 2.0 * x_i + 1.0
    diff_i = abs(pred_i - true_i)
    print(f"{x_i:<10.2f} | {pred_i:<22.4f} | {true_i:<20.4f} | {diff_i:<10.4f}")

print("=" * 55)
print("As expected, untrained model predictions are far from true values.")

---

## Cell 19 - Kiểm tra & Xác thực Model (Model Verification)

**WHAT**:
- Kiểm tra tính toàn vẹn của model: Shape, Device, Forward Pass trên batch thật từ `x_train`.

**WHY**:
- Đảm bảo model sẵn sàng tương thích 100% với dữ liệu đã chuẩn bị ở Phase 3 trước khi bước sang Phase 5 (Training Loop).

**HOW**:
- Kiểm tra shape input `[N, 1]` -> output `[N, 1]`.
- Kiểm tra việc chuyển model sang thiết bị (`device` CPU hoặc GPU/CUDA).
- Kiểm tra tính toán gradient khả dụng (`requires_grad=True`).

**WHEN**:
- Luôn chạy unit test / sanity check trước khi khởi động training loop lớn.

In [ ]:
# ============================================================
# CELL 19: Model Sanity Checks and Device Placement
# Purpose: Verify model compatibility with training dataset
# ============================================================

import sys
import os

IN_COLAB = 'google.colab' in sys.modules
PROJECT_ROOT = "/content/drive/MyDrive/AI_Training" if IN_COLAB else os.path.abspath(".")
PROCESSED_DIR = f"{PROJECT_ROOT}/data/processed"

print("=" * 55)
print("         MODEL VERIFICATION CHECKS")
print("=" * 55)

# --- Device selection ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Selected Device: {device}")

# Move model to device
model = model.to(device)

# --- Load a batch from processed train set ---
train_data = torch.load(f"{PROCESSED_DIR}/train/split.pt")
x_train_batch = train_data["x"][:32].to(device)  # First batch of 32 samples
y_train_batch = train_data["y"][:32].to(device)

# --- Perform Forward Pass on batch ---
batch_pred = model(x_train_batch)

# --- Sanity Checks ---
checks = []

# Check 1: Output shape matches [batch_size, 1]
check_shape = (batch_pred.shape == x_train_batch.shape == torch.Size([32, 1]))
checks.append(("Output shape matches [32, 1]", check_shape))

# Check 2: Output device matches model device
check_device = (batch_pred.device == device)
checks.append((f"Output device is {device}", check_device))

# Check 3: Parameters require gradient
check_grad = all(p.requires_grad for p in model.parameters())
checks.append(("Model parameters require grad for training", check_grad))

# Check 4: Output contains no NaN or Inf
check_values = not (torch.isnan(batch_pred).any() or torch.isinf(batch_pred).any())
checks.append(("Output contains no NaN or Inf", check_values))

# --- Display Check Results ---
all_passed = True
for name, passed in checks:
    status = "OK" if passed else "FAIL"
    print(f"  [{status}] {name}")
    if not passed:
        all_passed = False

print("-" * 55)
if all_passed:
    print("ALL CHECKS PASSED -- Model is fully ready for Phase 5 (Training)!")
else:
    print("Some checks FAILED. Please review model architecture.")
print("=" * 55)

---

## Tóm tắt Phase 4 - Model

### Bạn đã học được gì trong Phase 4:

| Thành phần | Khái niệm & Giải thích |
|---|---|
| **Linear Model** | Mô hình hồi quy tuyến tính học hàm số dạng $y = w \cdot x + b$. |
| **`torch.nn.Module`** | Base class chuẩn của PyTorch đóng gói architecture, parameters, và forward computation. |
| **Weight ($w$)** | Hệ số nhân của input $x$, thể hiện độ dốc của hàm dự đoán. Ban đầu là giá trị ngẫu nhiên. |
| **Bias ($b$)** | Hệ số dịch chuyển tự do, giúp đường thẳng không bị khóa cứng đi qua $(0, 0)$. |
| **Forward Pass** | Quá trình tính toán xuôi từ Input $x \rightarrow$ Model $\rightarrow$ Output $\hat{y}$. Luôn gọi `model(x)` thay vì `model.forward(x)`. |

---

### Luồng dữ liệu hoàn chỉnh đã sẵn sàng:

```
Dataset (Phase 3)              Model (Phase 4)              Training (Phase 5 Sắp tới)
─────────────────              ───────────────              ──────────────────────────
x_train [700, 1]  ────────►  model(x) = x * w + b  ──────►  y_pred [700, 1]
y_train [700, 1]                                                 │
                                                                 ▼
                                                        Loss = MSE(y_pred, y)
                                                                 │
                                                        loss.backward() (Gradient)
                                                                 │
                                                        optimizer.step() (Update w, b)
```

---

### Câu hỏi kiểm tra hiểu biết:

**Câu 1**: Nếu một mô hình Linear không có thành phần **Bias** ($b$) thì đồ thị đường thẳng $y = w \cdot x$ có hạn chế gì lớn nhất đối với bài toán $y = 2x + 1$?

**Câu 2**: Tại sao trong PyTorch ta luôn gọi `prediction = model(x)` mà không nên gọi trực tiếp `prediction = model.forward(x)`?

---

### Bài tập nhỏ thực hành:

1. Chạy các Cell 16, 17, 18, 19 trong notebook.
2. Thử thay đổi giá trị input ở Cell 18 (ví dụ $x = 50.0$) và quan sát kết quả dự đoán của model chưa qua huấn luyện.

---

**DỪNG TẠI ĐÂY.**

Khi bạn đã nắm vững các khái niệm trên, hãy nói **"Tiếp tục"** để chuyển sang **Phase 5 — Training**!

---
# Phase 5 — Training (Quá trình Huấn luyện)

**Mục tiêu của Phase 5**:
- Hiểu bản chất toán học và lập trình của quá trình huấn luyện AI.
- Nắm vững 5 bước cốt lõi: Forward -> Loss -> Zero Grad -> Backward -> Step.
- Hiểu Loss (MSE), Gradient, Backpropagation, Optimizer, Learning Rate, Epoch, Batch, DataLoader.
- Phân biệt `model.train()` và `model.eval()`, cùng vai trò bắt buộc của `torch.no_grad()` khi đánh giá.

### 1. Loss Function & Optimizer (Hàm Mất mát và Bộ Tối ưu hóa)

**WHAT**:
- **Loss Function (MSELoss)**: Đo lường độ sai lệch giữa giá trị dự đoán $\hat{y}$ và thực tế $y$.
  $$\text{MSE} = \frac{1}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)^2$$
- **Optimizer (SGD / Adam)**: Thuật toán cập nhật trọng số ($w, b$) dựa trên gradient để giảm Loss.
  $$w_{\text{new}} = w_{\text{old}} - \eta \cdot \frac{\partial L}{\partial w}$$
  trong đó $\eta$ là Learning Rate.

**WHY**:
- Nếu không có Loss, model không biết mình đang đoán đúng hay sai.
- Nếu không có Optimizer, model không biết phải điều chỉnh $w$ và $b$ theo hướng nào để tiến bộ.

**HOW**:
- `criterion = nn.MSELoss()`
- `optimizer = torch.optim.Adam(model.parameters(), lr=0.01)`

**WHEN**: Khởi tạo trước khi bước vào Training Loop.

In [ ]:
# ============================================================
# CELL 21: Loss Function and Optimizer Initialization
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from src.model import LinearRegressionModel
from src.utils import set_seed

set_seed(42)
model = LinearRegressionModel(input_size=1, output_size=1)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.05)

print("Model:", model)
print("Loss Function:", criterion)
print("Optimizer:", optimizer)


### 2. Chi tiết 1 bước học (Single Training Step)

Mỗi bước học diễn ra tuần tự 5 dòng code quan trọng bậc nhất:
1. `y_pred = model(x)`: **Forward Pass** — Tính toán đầu ra.
2. `loss = criterion(y_pred, y)`: **Compute Loss** — Tính sai số.
3. `optimizer.zero_grad()`: **Zero Gradients** — Xóa gradient cũ tích lũy từ bước trước.
4. `loss.backward()`: **Backpropagation** — Tính đạo hàm $\frac{\partial L}{\partial w}$ và $\frac{\partial L}{\partial b}$.
5. `optimizer.step()`: **Update Parameters** — Cập nhật $w$ và $b$ theo chiều dốc giảm của Loss.

In [ ]:
# ============================================================
# CELL 22: Single Training Step Detailed Walkthrough
# ============================================================
sample_x = torch.tensor([[2.0]], dtype=torch.float32)
sample_y = torch.tensor([[5.0]], dtype=torch.float32)  # 2*2 + 1 = 5

print(f"Initial Weight (w): {model.linear.weight.item():.4f}")
print(f"Initial Bias   (b): {model.linear.bias.item():.4f}")

# 1. Forward pass
y_pred = model(sample_x)
print(f"1. Forward Pass Prediction: {y_pred.item():.4f} (Expected: 5.0)")

# 2. Compute Loss
loss = criterion(y_pred, sample_y)
print(f"2. MSE Loss: {loss.item():.4f}")

# 3. Zero Grad
optimizer.zero_grad()

# 4. Backward Pass (Compute Gradients)
loss.backward()
print(f"4. Gradient dL/dw: {model.linear.weight.grad.item():.4f}")
print(f"   Gradient dL/db: {model.linear.bias.grad.item():.4f}")

# 5. Step (Update weights)
optimizer.step()
print(f"5. Updated Weight (w): {model.linear.weight.item():.4f}")
print(f"   Updated Bias   (b): {model.linear.bias.item():.4f}")


### 3. Vòng lặp huấn luyện hoàn chỉnh (Full Training Loop với DataLoader)

Trong thực tế, dữ liệu được chia theo **Batch** qua `DataLoader` và lặp lại qua nhiều **Epoch**.

In [ ]:
# ============================================================
# CELL 23: Complete Training Loop Execution
# ============================================================
from src.dataset import create_dataloaders
from src.evaluate import evaluate_model

train_loader, val_loader, test_loader, _ = create_dataloaders(project_root=".", batch_size=32)
epochs = 50
train_loss_history = []
val_loss_history = []

for epoch in range(1, epochs + 1):
    model.train()  # Set model to training mode
    running_loss = 0.0
    for x_b, y_b in train_loader:
        pred = model(x_b)
        loss = criterion(pred, y_b)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * len(x_b)
    
    epoch_train_loss = running_loss / len(train_loader.dataset)
    val_loss, val_mae, val_r2, _, _ = evaluate_model(model, val_loader, criterion, torch.device('cpu'))
    
    train_loss_history.append(epoch_train_loss)
    val_loss_history.append(val_loss)
    
    if epoch % 10 == 0 or epoch == 1:
        w = model.linear.weight.item()
        b = model.linear.bias.item()
        print(f"Epoch [{epoch:2d}/{epochs:2d}] | Train Loss: {epoch_train_loss:.6f} | Val Loss: {val_loss:.6f} | w: {w:.4f} (target 2.0) | b: {b:.4f} (target 1.0)")


---
# Phase 6 — Evaluation & Metrics (Đánh giá Mô hình)

**Mục tiêu của Phase 6**:
- Đánh giá mô hình trên tập Test chưa từng xuất hiện trong quá trình train.
- Hiểu các chỉ số: **MSE** (Mean Squared Error), **MAE** (Mean Absolute Error), **R²** (Coefficient of Determination).
- Đảm bảo luôn sử dụng `model.eval()` và `with torch.no_grad():`.

In [ ]:
# ============================================================
# CELL 24: Test Set Evaluation and Metrics
# ============================================================
from src.evaluate import evaluate_model, test_unseen_points

test_loss, test_mae, test_r2, y_true, y_pred = evaluate_model(model, test_loader, criterion, torch.device('cpu'))

print("=" * 55)
print("           FINAL TEST SET PERFORMANCE")
print("=" * 55)
print(f"Test MSE Loss : {test_loss:.6f}")
print(f"Test MAE      : {test_mae:.6f} (Mean Absolute Error)")
print(f"Test R² Score : {test_r2:.6f} (Target: 1.0 = Perfect fit)")
print("=" * 55)

# Evaluate custom unseen coordinates
test_unseen_points(model, torch.device('cpu'), custom_points=[11.0, 15.0, 20.0, 50.0, -15.0, 0.0])


In [ ]:
# ============================================================
# CELL 25: Visualization of Predictions vs Ground Truth
# ============================================================
import matplotlib.pyplot as plt
from src.evaluate import plot_predictions

plot_predictions(model, torch.device('cpu'), test_loader, save_path='plots/test_predictions.png')

# Display inline in notebook
img = plt.imread('plots/test_predictions.png')
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis('off')
plt.title('Prediction vs Ground Truth Verification')
plt.show()


---
# Phase 7 — Checkpoint & Resume Training (Lưu & Tiếp tục Huấn luyện)

**Mục tiêu của Phase 7**:
- Hiểu cách lưu trạng thái đầy đủ (`model_state_dict`, `optimizer_state_dict`, `scheduler_state_dict`, `epoch`, `best_val_loss`).
- Resume training khi Colab bị ngắt kết nối mà không cần train lại từ đầu.
- Phân biệt `best_model.pth` và `checkpoint_epoch_N.pth`.

In [ ]:
# ============================================================
# CELL 26: Save & Load Checkpoint Test
# ============================================================
from src.checkpoint import save_checkpoint, load_checkpoint, save_best_model

ckpt_file = 'models/checkpoints/checkpoint_epoch_50.pth'
save_checkpoint(
    path=ckpt_file,
    epoch=50,
    model=model,
    optimizer=optimizer,
    scheduler=None,
    loss=train_loss_history[-1],
    best_val_loss=val_loss_history[-1],
    config={'seed': 42, 'lr': 0.05}
)

# Simulate runtime disconnect by initializing a fresh model
fresh_model = LinearRegressionModel(input_size=1, output_size=1)
fresh_optimizer = optim.Adam(fresh_model.parameters(), lr=0.05)

start_epoch, last_loss, best_val, loaded_cfg = load_checkpoint(
    ckpt_file, fresh_model, fresh_optimizer, device=torch.device('cpu')
)

print(f'Resuming from Epoch: {start_epoch}')
print(f'Restored Model Weight: {fresh_model.linear.weight.item():.4f}')
print(f'Restored Model Bias  : {fresh_model.linear.bias.item():.4f}')


---
# Phase 8 — Stability: Early Stopping, LR Scheduler & Gradient Clipping

**Mục tiêu của Phase 8**:
- **Early Stopping**: Tự động dừng huấn luyện khi validation loss không giảm sau `patience` epochs để chống overfitting.
- **Learning Rate Scheduler (`ReduceLROnPlateau`)**: Giảm tốc độ học khi loss đi vào vùng bão hòa.
- **Gradient Clipping (`clip_grad_norm_`)**: Chặn hiện tượng bùng nổ gradient (Exploding Gradients).

In [ ]:
# ============================================================
# CELL 27: Training with Early Stopping & LR Scheduler
# ============================================================
from torch.optim.lr_scheduler import ReduceLROnPlateau

model_stable = LinearRegressionModel()
optimizer_stable = optim.Adam(model_stable.parameters(), lr=0.1)
scheduler = ReduceLROnPlateau(optimizer_stable, mode='min', patience=3, factor=0.5)

patience = 5
best_val = float('inf')
no_improve = 0

print('Starting stable training loop with patience = 5...')
for ep in range(1, 40):
    model_stable.train()
    for xb, yb in train_loader:
        pred = model_stable(xb)
        l = criterion(pred, yb)
        optimizer_stable.zero_grad()
        l.backward()
        torch.nn.utils.clip_grad_norm_(model_stable.parameters(), max_norm=1.0)
        optimizer_stable.step()
    
    val_l, _, _, _, _ = evaluate_model(model_stable, val_loader, criterion, torch.device('cpu'))
    scheduler.step(val_l)
    
    if val_l < best_val:
        best_val = val_l
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f'Early stopping triggered at epoch {ep}! Best Val Loss: {best_val:.6f}')
            break


---
# Phase 9 — Experiment Tracking (Quản lý Thí nghiệm)

**Mục tiêu của Phase 9**:
- Tự động hóa toàn bộ quá trình chạy experiment qua module `src/train.py`.
- Lưu cấu hình bất biến (`config.yaml`), nhật ký huấn luyện (`train.log`), bảng đo lường (`metrics.csv`), và đồ thị (`loss.png`).

In [ ]:
# ============================================================
# CELL 28: Run Experiment 001 Pipeline
# ============================================================
from src.train import train_model

exp_results = train_model(
    config_path='configs/config_v1.yaml',
    experiment_name='experiment_001',
    project_root='.'
)

print('\nExperiment 001 completed successfully!')
print(f'Results: {exp_results}')


---
# Phase 10 — Scaling (Mở rộng Dữ liệu và Quy mô)

**Mục tiêu của Phase 10**:
- Khả năng scale dữ liệu từ 1,000 -> 10,000 -> 100,000 samples mà không cần đổi kiến trúc code.
- Quản lý dataset versions (`dataset_v1`, `dataset_v2`).

In [ ]:
# ============================================================
# CELL 29: Generate Large Dataset (10,000 samples)
# ============================================================
from src.dataset import generate_and_save_dataset

generate_and_save_dataset(
    project_root='.',
    version='v2',
    num_samples=10000,
    seed=42
)


---
# Phase 11 — Advanced AI: Neural Networks & Math/Science Solver

**Mục tiêu của Phase 11**:
- Chuyển tiếp từ Linear Model sang **Multi-Layer Perceptron (Neural Network)** để giải bài toán phi tuyến: $y = 2x^2 - 3x + 1$.
- Kiến trúc mạng Deep Learning mở rộng cho CNN (thị giác), NLP / Transformer (xử lý ngôn ngữ), LLM Fine-tuning (LoRA/QLoRA), và AI Gia sư Toán/Lý/Hóa.

In [ ]:
# ============================================================
# CELL 30: Train Multi-Layer Perceptron on Non-linear Formula
# Formula: y = 2x^2 - 3x + 1 + noise
# ============================================================
from src.advanced import train_advanced_nn_experiment

test_r2 = train_advanced_nn_experiment(project_root='.', epochs=60, lr=0.01)
print(f'\nAdvanced Non-linear MLP Model achieved R² Score: {test_r2:.4f}')


---
## Project Completion Summary (Tổng kết Dự án)

Toàn bộ 11 Phases đã được triển khai hoàn chỉnh:
- **Phase 1**: Environment, Google Colab & Google Drive, CUDA/GPU, Random Seed.
- **Phase 2**: Tensors, Shapes, Dtypes, Device mapping.
- **Phase 3**: Dataset Generation & Versioning, Train/Validation/Test Split.
- **Phase 4**: Linear Model, Weights, Bias, Forward Pass.
- **Phase 5**: Training Loop, MSE Loss, Backpropagation, Optimizer, `model.train()` / `model.eval()`.
- **Phase 6**: Evaluation Metrics (MSE, MAE, R²), Unseen coordinate predictions, Visualization.
- **Phase 7**: Checkpoint Save/Load, Resume Training, Best Model.
- **Phase 8**: Early Stopping, LR Scheduler, Gradient Clipping.
- **Phase 9**: Experiment Tracking (`config.yaml`, `train.log`, `metrics.csv`, `loss.png`).
- **Phase 10**: Scaling to 10,000+ samples and large datasets.
- **Phase 11**: Multi-Layer Perceptron (MLP) for non-linear equations & roadmap to Transformer, LoRA, and Math/Science AI Tutor.